# 10a. Shannon Entropy Reranking — Base (Facial Skincare)

This notebook evaluates the deterministic Base condition on the fixed query-only candidate pool for the 2,288-case benchmark. User-profile, history-concentration, brand-affinity, and prior-recency components are disabled. The personalization weight is fixed at zero, so the incoming candidate order is preserved.

Evaluation is repeated at the five fixed candidate depths. The notebook reports per-query and aggregate ranking metrics, runtime summaries, candidate-level scores and ranks, and feature-contract, temporal, and leakage-control diagnostics.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Environment, paths, and config

In [ ]:
import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

print("Libraries loaded.")
print("Random seed:", RANDOM_SEED)


In [ ]:
# =========================================================
# Config
# =========================================================
NOTEBOOK_NAME = '10a_no_prior_rerank_shannon_entropy_face.ipynb'
CATEGORY_ID = 'face'
CATEGORY_FOLDER = 'facial_skincare'
CATEGORY_LABEL = 'Facial Skincare'
EXPERIMENT_CONDITION = 's2q_no_prior_reranking'
CANDIDATE_POOL_ROLE = 'baseline_query_only'
USE_USER_PRIOR_FEATURES = False
USER_PRIOR_FEATURE_POLICY = "all_strict_pre_target_prior" if USE_USER_PRIOR_FEATURES else "no_user_prior_features"
SHANNON_FRAMEWORK_VERSION = "winner_aware_shannon_v3_reviewed"
STAGE1_CANDIDATE_CONTRACT = "notebook09_winner_long_pool_exact_k"
ITEM_FACET_EVIDENCE_POLICY = "metadata_core_functional_plus_brand_no_review_derived"
ENTROPY_NORMALIZATION_POLICY = "global_catalog_vocabulary_normalized_entropy_unchanged"
SHANNON_SCORING_POLICY = "within_query_minmax_retrieval_plus_entropy_gated_profile_match"
ALGORITHM_CHANGE_SCOPE = "winner_routing_and_leakage_guardrails_only_no_retuning"
CANDIDATE_SOURCE_LABEL = "resolved_from_notebook09"
CANDIDATE_SOURCE_FORMAT = "notebook09_winner_long_pool"
STAGE = 'stage2_baseline_retrieval_no_prior_rerank_shannon'

RERANKING_METHOD = 'shannon_no_prior'
SHANNON_OUTPUT_RERANK_METHOD = 'shannon_no_prior_rerank'
BASELINE_RERANK_METHOD = "stage1_baseline"
EXPECTED_RERANK_METHODS = [BASELINE_RERANK_METHOD, SHANNON_OUTPUT_RERANK_METHOD]

STAGE1_QUERY_METHOD = "C"
QUERY_METHOD = STAGE1_QUERY_METHOD
QUERY_TEXT_COL = "query"
QUERY_VARIANT = "C"

# Retrieval lineage is resolved from the Notebook 09 winner manifest after it is loaded.


POOL_K = 1000
POOL_DEPTHS = [100, 300, 500, 700, 1000]
REPORT_POOL_DEPTH = 1000
RUNTIME_REPORT_POOL_DEPTH = REPORT_POOL_DEPTH
EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]

PRIMARY_STAGE2_METRIC = "NDCG@5"
PRIMARY_STAGE2_METRICS = ["NDCG@5"]
SECONDARY_STAGE2_METRICS = ["HitRate@5", "NDCG@1", "HitRate@1", "MRR@5"]
SUPPLEMENTARY_STAGE2_METRICS = ["NDCG@10", "HitRate@10", "MRR@10"]
COMMON_STAGE2_COMPARISON_METRICS = [
    "NDCG@5", "HitRate@5", "NDCG@1", "HitRate@1", "MRR@5",
    "NDCG@10", "HitRate@10", "MRR@10",
]
PREFERENCE_ALIGNMENT_K = 5
PREFERENCE_DIAGNOSTIC_METRICS = ['weighted_facet_overlap_at_5', 'brand_match_at_5', 'concern_match_at_5', 'ingredient_match_at_5']
PREFERENCE_DIAGNOSTIC_FAMILIES = ['brand', 'concern', 'ingredient']
PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS = {'brand': 1.0, 'concern': 1.0, 'ingredient': 1.0}
UPLIFT_DIAGNOSTIC_METRICS = PREFERENCE_DIAGNOSTIC_METRICS

SHANNON_ALPHA = 0.0
SHANNON_HISTORY_SATURATION_N = 8
REGIME_STRENGTH_CAP = {'cold': 0.0, 'weak': 0.2, 'moderate': 0.45, 'strong': 0.8}
REGIME_ORDER = ['cold', 'weak', 'moderate', 'strong']
DEDUP_HISTORY_BY_CASE_ITEM = False

BRAND_QUERY_ENABLED = False
BRAND_CANDIDATE_VISIBLE = True
BRAND_RERANKING_ENABLED = bool(USE_USER_PRIOR_FEATURES)
USER_BRAND_AFFINITY_ENABLED = USE_USER_PRIOR_FEATURES
HISTORY_SOURCE = "all_prior" if USE_USER_PRIOR_FEATURES else "none"
HISTORICAL_POPULATION_REVIEW_SIGNALS_IN_USER_PROFILE = False
SHARED_ALL_PRIOR_CONTRACT_ROLE = "not_applicable"

NO_PRIOR_ABLATION = True
NO_PRIOR_FEATURE_POLICY = "Base: Notebook 09 query-only winner + deterministic no-prior Shannon control"
NO_PRIOR_SCORE_POLICY = "All user-profile, profile-concentration, and user-history temporal score components are disabled."
NO_PRIOR_MANIFEST_NOTE = NO_PRIOR_SCORE_POLICY
NO_PRIOR_SHANNON_MANIFEST_NOTE = NO_PRIOR_SCORE_POLICY
NO_PRIOR_SCORE_FEATURES = ["rank", "score"]
NO_PRIOR_SHANNON_FINAL_SCORE_USES_PERSONALIZATION = False
NO_PRIOR_SHANNON_FINAL_SCORE_USES_TEMPORAL_RECENCY = False


EXPECTED_QUERY_COUNT = None
EXPECTED_CANDIDATE_ROWS_TOP1000 = None
EXPECTED_REGIME_COUNTS = {}
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
STRICT_SAMPLE_COUNT_QC = True

RUNTIME_BRANCH = 'baseline_winner_shannon_no_prior'
RUNTIME_ROWS = []
NOTEBOOK_TIMER_START = time.perf_counter()

PROJECT_ROOT = Path('/content/drive/MyDrive/thesis_recsys/categories/facial_skincare')
os.chdir(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
output_paths = {}

STAGE1_POOL_DIR = OUTPUTS_DIR / "stage1_candidate_pools"
CANDIDATE_POOL_MANIFEST_PATH = STAGE1_POOL_DIR / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"
CANDIDATE_POOL_SUMMARY_PATH = STAGE1_POOL_DIR / f"winner_candidate_pool_summary_{CATEGORY_ID}.csv"
CANDIDATE_POOL_PATH = None
QUERY_CACHE_PARQUET = None

STAGE_OUTPUT_DIR = OUTPUTS_DIR / 'stage2_nonpersonalized_rerank/shannon_no_prior'
STAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = STAGE_OUTPUT_DIR

QUERY_CACHE_SUMMARY_PATH = PROJECT_ROOT / 'outputs/query_summary/face_queries_abcd_summary.csv'
QUERY_CACHE_CONFIG_PATH = PROJECT_ROOT / 'outputs/query_summary/face_queries_abcd_config.json'
PRIOR_HISTORY_PARQUET = PROJECT_ROOT / 'data/processed/user_sampling/face_user_prior_review_history.parquet'
ITEM_SCHEMA_PARQUET = PROJECT_ROOT / 'data/processed/items/face_item_schema_full.parquet'
ITEM_SCHEMA_BASE_PARQUET = PROJECT_ROOT / 'data/processed/items/face_item_schema.parquet'
ITEM_DOCS_PARQUET = PROJECT_ROOT / 'data/processed/items/face_item_docs.parquet'
ITEMS_FACETS_PARQUET = PROJECT_ROOT / 'data/processed/items/face_items_facets.parquet'
PROCESSED_ITEMS_DIR = ITEM_SCHEMA_PARQUET.parent

PERSONALIZED_RETRIEVAL_DIR = OUTPUTS_DIR / "stage1_personalized_retrieval"
SOURCE_PER_QUERY_METRICS_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_per_query_metrics_{CATEGORY_ID}.parquet"
SOURCE_RESULTS_BY_REGIME_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_results_by_regime_{CATEGORY_ID}.csv"
SOURCE_CONFIG_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_config_snapshot_{CATEGORY_ID}.json"
SOURCE_MANIFEST_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_manifest_{CATEGORY_ID}.json"


print("Notebook:", NOTEBOOK_NAME)
print("Project root:", PROJECT_ROOT)
print("Experiment condition:", EXPERIMENT_CONDITION)
print("Candidate pool role:", CANDIDATE_POOL_ROLE)
print("User-prior features:", USE_USER_PRIOR_FEATURES)
print("Notebook 09 manifest:", CANDIDATE_POOL_MANIFEST_PATH)
print("Output directory:", OUT_DIR)
print("POOL_DEPTHS:", POOL_DEPTHS)
print("EVAL_KS:", EVAL_KS)


In [ ]:
# =========================================================
# Temporal Artifact Config for Shannon Heuristic
# =========================================================
TEMPORAL_FEATURE_VERSION = "temporal_recency_v2_compact_prequery_only"
SHANNON_FEATURE_REGISTRY_VERSION = "face_shannon_compact_components_v2"
USE_TEMPORAL_RECENCY_FEATURES = True
SHANNON_TEMPORAL_RECENCY_WEIGHT = 0.0
TEMPORAL_WINDOWS_DAYS = [30, 90, 180]
MS_PER_DAY = 1000 * 60 * 60 * 24
RECENCY_FEATURE_FILL_DAYS = np.float32(9999.0)
TEMPORAL_LEAKAGE_RULE = "review_timestamp_ms < target_timestamp_ms"

TEMPORAL_ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed" / "temporal"
ITEM_REVIEW_TIME_INDEX_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_review_timestamp_index.parquet"
ITEM_REVIEW_DAILY_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_daily_review_counts.parquet"
ENTITY_REVIEW_DAILY_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_entity_daily_review_counts.parquet"
ITEM_TEMPORAL_SUMMARY_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_temporal_summary.parquet"
TEMPORAL_ARTIFACT_MANIFEST_PATH = TEMPORAL_ARTIFACT_DIR / "temporal_artifact_manifest_face.json"

TEMPORAL_ARTIFACT_PATHS = {
    "item_review_time_index": ITEM_REVIEW_TIME_INDEX_PATH,
    "item_review_daily_counts": ITEM_REVIEW_DAILY_COUNTS_PATH,
    "entity_review_daily_counts": ENTITY_REVIEW_DAILY_COUNTS_PATH,
    "item_temporal_summary": ITEM_TEMPORAL_SUMMARY_PATH,
    "temporal_artifact_manifest": TEMPORAL_ARTIFACT_MANIFEST_PATH,
}

FORBIDDEN_MODEL_FEATURE_COLS = {
    "timestamp_ms", "target_timestamp_ms", "case_id", "query_id", "user_id",
    "target_parent_asin", "candidate_parent_asin", "target_item_id", "candidate_item_id",
    "gt_item_id", "is_gt", "label", "target_rank_desc", "target_selection_mode",
    "query_text", "review_text", "review_body", "raw_review_text", "target_review_text", "heldout_review_text",
}

print("SHANNON_FEATURE_REGISTRY_VERSION:", SHANNON_FEATURE_REGISTRY_VERSION)
print("TEMPORAL_FEATURE_VERSION:", TEMPORAL_FEATURE_VERSION)
print("Shannon temporal recency weight:", SHANNON_TEMPORAL_RECENCY_WEIGHT)

## 2. Runtime Measurement

The notebook records feature preparation, scoring, evaluation, export, and total wall-clock components in a common runtime schema.


In [ ]:
def record_runtime_step(
    step_name,
    runtime_sec,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type='wall_clock_step',
    derived_from_existing_runtime=False,
    error='',
    **extra_metadata,
):
    n_queries_value = np.nan if n_queries is None else n_queries
    n_candidates_value = np.nan if n_candidates is None else n_candidates
    runtime_sec_value = np.nan if runtime_sec is None else float(runtime_sec)

    row = {
        'category': CATEGORY_FOLDER,
        'category_id': CATEGORY_ID,
        'notebook_name': NOTEBOOK_NAME,
        'method': method or RERANKING_METHOD,
        'branch': branch or RUNTIME_BRANCH,
        'step_name': step_name,
        'pool_depth': pool_depth,
        'n_queries': n_queries_value,
        'n_candidates': n_candidates_value,
        'runtime_sec': runtime_sec_value,
        'runtime_sec_per_query': runtime_sec_value / n_queries_value if pd.notna(runtime_sec_value) and pd.notna(n_queries_value) and n_queries_value else np.nan,
        'runtime_sec_per_candidate': runtime_sec_value / n_candidates_value if pd.notna(runtime_sec_value) and pd.notna(n_candidates_value) and n_candidates_value else np.nan,
        'error': error,
        'runtime_measurement_type': runtime_measurement_type,
        'derived_from_existing_runtime': bool(derived_from_existing_runtime),
    }
    row.update(extra_metadata)
    RUNTIME_ROWS.append(row)
    return row


@contextmanager
def runtime_step(
    step_name,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type='wall_clock_step',
    derived_from_existing_runtime=False,
    **extra_metadata,
):
    start_time = time.perf_counter()
    error_message = ''
    try:
        yield
    except Exception as exc:
        error_message = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        record_runtime_step(
            step_name,
            time.perf_counter() - start_time,
            method=method,
            branch=branch,
            pool_depth=pool_depth,
            n_queries=n_queries,
            n_candidates=n_candidates,
            runtime_measurement_type=runtime_measurement_type,
            derived_from_existing_runtime=derived_from_existing_runtime,
            error=error_message,
            **extra_metadata,
        )


def _runtime_sum(runtime_steps_df, step_names, pool_depth=REPORT_POOL_DEPTH, method=RERANKING_METHOD):
    if runtime_steps_df is None or runtime_steps_df.empty:
        return 0.0
    rows = runtime_steps_df.copy()
    if 'method' in rows.columns:
        rows = rows[rows['method'].astype(str).eq(str(method))]
    if 'pool_depth' in rows.columns:
        rows = rows[pd.to_numeric(rows['pool_depth'], errors='coerce').eq(int(pool_depth))]
    rows = rows[rows['step_name'].astype(str).isin(list(step_names))]
    vals = pd.to_numeric(rows['runtime_sec'], errors='coerce').dropna()
    return float(vals.sum()) if len(vals) else 0.0


def _metric_from_summary(summary_df, method_name, metric_col):
    if summary_df is None or summary_df.empty or metric_col not in summary_df.columns:
        return np.nan
    rows = summary_df.copy()
    if 'rerank_method' in rows.columns:
        rows = rows[rows['rerank_method'].astype(str).eq(method_name)]
    if rows.empty:
        return np.nan
    vals = pd.to_numeric(rows[metric_col], errors='coerce').dropna()
    return float(vals.iloc[0]) if len(vals) else np.nan


def export_runtime_logs(output_dir, report_pool_depth=REPORT_POOL_DEPTH):
    output_dir = Path(output_dir)
    export_start = time.perf_counter()

    n_queries_at_report = np.nan
    n_candidates_at_report = np.nan
    if 'per_query_report_df' in globals() and isinstance(per_query_report_df, pd.DataFrame) and not per_query_report_df.empty:
        report_rows = per_query_report_df[per_query_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]
        n_queries_at_report = float(report_rows['query_id'].nunique()) if 'query_id' in report_rows.columns else float(len(report_rows))
        n_candidates_at_report = n_queries_at_report * int(report_pool_depth) if pd.notna(n_queries_at_report) else np.nan
    if 'reranked_candidates_report_df' in globals() and isinstance(reranked_candidates_report_df, pd.DataFrame) and not reranked_candidates_report_df.empty:
        n_candidates_at_report = float(len(reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]))

    # Add export and total rows after all material outputs have been written.
    # If the material-output cell already recorded export_outputs, avoid a duplicate step row.
    export_runtime_sec = time.perf_counter() - export_start
    existing_export_row = any(
        str(row.get('step_name', '')) == 'export_outputs'
        and int(row.get('pool_depth') or report_pool_depth) == int(report_pool_depth)
        for row in RUNTIME_ROWS
    )
    if not existing_export_row:
        record_runtime_step(
            'export_outputs',
            export_runtime_sec,
            pool_depth=report_pool_depth,
            n_queries=n_queries_at_report,
            n_candidates=n_candidates_at_report,
        )
    total_runtime_sec = time.perf_counter() - NOTEBOOK_TIMER_START
    record_runtime_step(
        'total_notebook',
        total_runtime_sec,
        pool_depth=report_pool_depth,
        n_queries=n_queries_at_report,
        n_candidates=n_candidates_at_report,
        runtime_measurement_type='wall_clock_total',
    )

    runtime_steps_df = pd.DataFrame(RUNTIME_ROWS)
    if not runtime_steps_df.empty:
        runtime_steps_df['pool_depth'] = pd.to_numeric(runtime_steps_df['pool_depth'], errors='coerce')

    online_steps = ['feature_preparation', 'model_scoring', 'ranking_sorting']
    offline_steps = ['model_fit_or_tuning']
    evaluation_export_steps = ['evaluation', 'export_outputs']

    online_runtime_sec = _runtime_sum(runtime_steps_df, online_steps, report_pool_depth)
    offline_runtime_sec = _runtime_sum(runtime_steps_df, offline_steps, report_pool_depth)
    evaluation_export_runtime_sec = _runtime_sum(runtime_steps_df, evaluation_export_steps, report_pool_depth)

    ndcg_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'NDCG@5') if 'summary_overall_df' in globals() else np.nan
    hitrate_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'HitRate@5') if 'summary_overall_df' in globals() else np.nan
    mrr_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'MRR@5') if 'summary_overall_df' in globals() else np.nan
    baseline_ndcg_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, 'NDCG@5') if 'summary_overall_df' in globals() else np.nan
    baseline_hitrate_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, 'HitRate@5') if 'summary_overall_df' in globals() else np.nan
    delta_ndcg_at_5 = ndcg_at_5 - baseline_ndcg_at_5 if pd.notna(ndcg_at_5) and pd.notna(baseline_ndcg_at_5) else np.nan
    delta_hitrate_at_5 = hitrate_at_5 - baseline_hitrate_at_5 if pd.notna(hitrate_at_5) and pd.notna(baseline_hitrate_at_5) else np.nan

    runtime_method_summary_at1000_df = pd.DataFrame([{
        'category': CATEGORY_FOLDER,
        'category_id': CATEGORY_ID,
        'notebook_name': NOTEBOOK_NAME,
        'branch': RUNTIME_BRANCH,
        'method': RERANKING_METHOD,
        'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
        'pool_depth': int(report_pool_depth),
        'n_queries': n_queries_at_report,
        'n_candidates': n_candidates_at_report,
        'online_operation_runtime_sec': online_runtime_sec,
        'offline_preparation_runtime_sec': offline_runtime_sec,
        'evaluation_export_runtime_sec': evaluation_export_runtime_sec,
        'total_notebook_runtime_sec': total_runtime_sec,
        'runtime_sec_per_query': online_runtime_sec / n_queries_at_report if pd.notna(n_queries_at_report) and n_queries_at_report else np.nan,
        'runtime_sec_per_candidate': online_runtime_sec / n_candidates_at_report if pd.notna(n_candidates_at_report) and n_candidates_at_report else np.nan,
        'queries_per_second': n_queries_at_report / online_runtime_sec if pd.notna(n_queries_at_report) and online_runtime_sec else np.nan,
        'candidates_per_second': n_candidates_at_report / online_runtime_sec if pd.notna(n_candidates_at_report) and online_runtime_sec else np.nan,
        'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
        'candidate_pool_type': candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else 'resolved_from_notebook09',
        'retrieval_method': retrieval_method_actual if 'retrieval_method_actual' in globals() else 'resolved_from_notebook09',
        'retrieval_method_label': retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else 'resolved_from_notebook09',
        'reranking_method': RERANKING_METHOD,
        'sample_scope': 'native',
        'common_sample_filtering_introduced': False,
        'primary_metric': PRIMARY_STAGE2_METRIC,
        'ndcg_at_5': ndcg_at_5,
        'hitrate_at_5': hitrate_at_5,
        'mrr_at_5': mrr_at_5,
        'baseline_ndcg_at_5': baseline_ndcg_at_5,
        'baseline_hitrate_at_5': baseline_hitrate_at_5,
        'delta_ndcg_at_5_vs_baseline': delta_ndcg_at_5,
        'delta_hitrate_at_5_vs_baseline': delta_hitrate_at_5,
        'delta_ndcg5_per_100sec_online': (delta_ndcg_at_5 / online_runtime_sec) * 100 if pd.notna(delta_ndcg_at_5) and online_runtime_sec else np.nan,
        'delta_hitrate5_per_100sec_online': (delta_hitrate_at_5 / online_runtime_sec) * 100 if pd.notna(delta_hitrate_at_5) and online_runtime_sec else np.nan,
    }])

    runtime_notebook_summary_df = runtime_method_summary_at1000_df.copy()
    runtime_notebook_summary_df['total_logged_step_runtime_sec'] = pd.to_numeric(runtime_steps_df.get('runtime_sec', pd.Series(dtype='float64')), errors='coerce').sum()

    component_cols = ['feature_preparation', 'model_fit_or_tuning', 'model_scoring', 'ranking_sorting', 'evaluation', 'export_outputs']
    runtime_method_components_at1000_df = pd.DataFrame([{
        'category_id': CATEGORY_ID,
        'category_folder': CATEGORY_FOLDER,
        'method': RERANKING_METHOD,
        'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
        'pool_depth': int(report_pool_depth),
        'n_queries': n_queries_at_report,
        'feature_preparation_runtime_sec': _runtime_sum(runtime_steps_df, ['feature_preparation'], report_pool_depth),
        'model_fit_or_tuning_runtime_sec': _runtime_sum(runtime_steps_df, ['model_fit_or_tuning'], report_pool_depth),
        'model_scoring_runtime_sec': _runtime_sum(runtime_steps_df, ['model_scoring'], report_pool_depth),
        'ranking_sorting_runtime_sec': _runtime_sum(runtime_steps_df, ['ranking_sorting'], report_pool_depth),
        'evaluation_runtime_sec': _runtime_sum(runtime_steps_df, ['evaluation'], report_pool_depth),
        'export_outputs_runtime_sec': _runtime_sum(runtime_steps_df, ['export_outputs'], report_pool_depth),
        'online_operation_runtime_sec': online_runtime_sec,
        'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
        'candidate_pool_type': candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else 'resolved_from_notebook09',
        'retrieval_method': retrieval_method_actual if 'retrieval_method_actual' in globals() else 'resolved_from_notebook09',
        'retrieval_method_label': retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else 'resolved_from_notebook09',
        'reranking_method': RERANKING_METHOD,
    }])

    if 'runtime_by_pool_depth_df' in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
        runtime_pool_depth_diagnostic_df = runtime_by_pool_depth_df.copy()
    else:
        runtime_pool_depth_diagnostic_df = pd.DataFrame([{
            'category_id': CATEGORY_ID,
            'method': RERANKING_METHOD,
            'reranker': SHANNON_OUTPUT_RERANK_METHOD,
            'pool_depth': int(report_pool_depth),
            'n_queries': n_queries_at_report,
            'n_candidates': n_candidates_at_report,
        }])
    runtime_pool_depth_diagnostic_df['notebook_name'] = NOTEBOOK_NAME
    runtime_pool_depth_diagnostic_df['branch'] = RUNTIME_BRANCH
    runtime_pool_depth_diagnostic_df['summary_scope'] = 'all_pool_depths'
    runtime_pool_depth_diagnostic_df['sample_scope'] = 'native'
    runtime_pool_depth_diagnostic_df['candidate_pool_type'] = candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else 'resolved_from_notebook09'
    runtime_pool_depth_diagnostic_df['retrieval_method'] = retrieval_method_actual if 'retrieval_method_actual' in globals() else 'resolved_from_notebook09'
    runtime_pool_depth_diagnostic_df['retrieval_method_label'] = retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else 'resolved_from_notebook09'
    runtime_pool_depth_diagnostic_df['reranking_method'] = RERANKING_METHOD

    runtime_steps_df.to_csv(output_dir / 'runtime_steps.csv', index=False)
    runtime_notebook_summary_df.to_csv(output_dir / 'runtime_notebook_summary.csv', index=False)
    runtime_method_summary_at1000_df.to_csv(output_dir / 'runtime_method_summary_at1000.csv', index=False)
    runtime_pool_depth_diagnostic_df.to_csv(output_dir / 'runtime_pool_depth_diagnostic.csv', index=False)
    runtime_method_components_at1000_df.to_csv(output_dir / 'runtime_method_components_at1000_face.csv', index=False)

    return (
        runtime_steps_df,
        runtime_notebook_summary_df,
        runtime_method_summary_at1000_df,
        runtime_pool_depth_diagnostic_df,
        runtime_method_components_at1000_df,
    )

print('Runtime framework ready.')

## 3. Shared helpers and schema standardization

In [ ]:
# Exact-contract helpers

def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8-sig") as handle:
        return json.load(handle)


def load_json_required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required JSON artifact: {path}")
    with open(path, "r", encoding="utf-8-sig") as handle:
        return json.load(handle)


def make_jsonable(obj):
    if isinstance(obj, dict):
        return {str(key): make_jsonable(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [make_jsonable(value) for value in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.bool_):
        return bool(obj)
    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass
    return obj


def require_columns(frame, columns, label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def normalize_space(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def normalize_text(value):
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9+\s_-]", " ", text)
    return re.sub(r"\s+", " ", text.replace("_", " ")).strip()


def normalize_item_id(value):
    return normalize_space(value)


def as_bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).ne(0)
    return series.fillna("").astype(str).str.strip().str.lower().isin({"1", "true", "t", "yes", "y"})


def to_timestamp_ms(series):
    values = pd.to_numeric(series, errors="coerce")
    positive = values[values.gt(0)]
    if not positive.empty and float(positive.median()) < 1e11:
        values = values * 1000.0
    return values


def safe_numeric(series, default=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(default)


def safe_int_series(series, default=0):
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(int)


def normalize_regime(value):
    value = normalize_text(value)
    return value if value in set(REGIME_ORDER) else ""


def regime_from_prior_count(prior_n):
    prior_n = int(float(prior_n)) if pd.notna(prior_n) else 0
    if prior_n <= 0:
        return "cold"
    if CATEGORY_ID == "face" and prior_n <= 3:
        return "weak"
    if CATEGORY_ID == "face" and prior_n <= 9:
        return "moderate"
    return "strong"


def apply_regime_order(frame, regime_col="regime"):
    if regime_col not in frame.columns:
        return frame
    out = frame.copy()
    values = out[regime_col].map(normalize_regime)
    out[regime_col] = pd.Categorical(values, categories=REGIME_ORDER, ordered=True)
    return out


def minmax_by_group(frame, group_col, value_col, rank_col="rank", max_rank=POOL_K):
    values = pd.to_numeric(frame[value_col], errors="coerce").fillna(0.0)
    group_min = values.groupby(frame[group_col]).transform("min")
    group_max = values.groupby(frame[group_col]).transform("max")
    denominator = (group_max - group_min).replace(0, np.nan)
    normalized = (values - group_min) / denominator
    rank_values = pd.to_numeric(frame[rank_col], errors="coerce").fillna(max_rank).clip(lower=1)
    rank_fallback = 1.0 - ((rank_values - 1.0) / max(max_rank - 1, 1))
    return normalized.where(denominator.notna(), rank_fallback).fillna(0.0).clip(0.0, 1.0)


def metrics_at_rank(rank, ks):
    rank_value = int(rank) if pd.notna(rank) else None
    row = {}
    for k in ks:
        hit = int(rank_value is not None and rank_value <= int(k))
        row[f"HitRate@{k}"] = float(hit)
        row[f"NDCG@{k}"] = float(1.0 / math.log2(rank_value + 1)) if hit else 0.0
        row[f"MRR@{k}"] = float(1.0 / rank_value) if hit else 0.0
    return row


def jaccard(left, right):
    left = set(left or [])
    right = set(right or [])
    return float(len(left & right) / len(left | right)) if left and right else 0.0


print("Exact-contract helpers ready.")


In [ ]:
# Notebook 09 exact candidate schema is enforced in the next input cell.
REQUIRED_NOTEBOOK09_COLUMNS = [
    "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "target_parent_asin", "gt_item_id", "target_timestamp_ms",
    "prior_history_n", "query_text", "candidate_pool_role", "method_slug",
    "retrieval_method", "candidate_parent_asin", "candidate_rank", "candidate_score",
    "candidate_score_source", "is_gt",
]

print("Notebook 09 exact candidate schema registered.")


## 4. Validate fixed paths and load Stage 1 candidate pool

In [ ]:
# =========================================================
# Validate Static Inputs
# =========================================================
required_input_paths = {
    "stage1_candidate_pool_manifest": CANDIDATE_POOL_MANIFEST_PATH,
    "items_facets_parquet": ITEMS_FACETS_PARQUET,
}
if USE_USER_PRIOR_FEATURES:
    required_input_paths["prior_history_parquet"] = PRIOR_HISTORY_PARQUET

optional_input_paths = {
    "candidate_pool_summary": CANDIDATE_POOL_SUMMARY_PATH,
    "item_schema_parquet": ITEM_SCHEMA_PARQUET,
    "item_schema_base_parquet": ITEM_SCHEMA_BASE_PARQUET,
    "item_docs_parquet": ITEM_DOCS_PARQUET,
    "query_cache_summary_path": QUERY_CACHE_SUMMARY_PATH,
    "query_cache_config_path": QUERY_CACHE_CONFIG_PATH,
}
missing_inputs = [name for name, path in required_input_paths.items() if not Path(path).exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("Static input paths validated.")
for name, path in {**required_input_paths, **optional_input_paths}.items():
    print(f"{name}: {path} | exists={Path(path).exists()}")


In [ ]:
# =========================================================
# Load the exact Notebook 09 winner candidate pool
# =========================================================
load_start = time.perf_counter()

stage1_pool_manifest = load_json_required(CANDIDATE_POOL_MANIFEST_PATH)
if stage1_pool_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 09 candidate manifest category mismatch.")
if stage1_pool_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 09 must use the exact-K candidate budget policy.")
if stage1_pool_manifest.get("variable_candidate_count_allowed") is not False:
    raise RuntimeError("Variable candidate counts are not allowed for Stage 2.")

output_paths_09 = stage1_pool_manifest.get("output_paths", {})
input_paths_09 = stage1_pool_manifest.get("input_paths", {})
if CANDIDATE_POOL_ROLE == "baseline_query_only":
    candidate_path_key = "query_only_winner_long"
    CANDIDATE_RETRIEVAL_METHOD_KEY = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_key"))
    CANDIDATE_RETRIEVAL_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_label"))
elif CANDIDATE_POOL_ROLE == "personalized_retrieval":
    candidate_path_key = "personalized_winner_long"
    CANDIDATE_RETRIEVAL_METHOD_KEY = normalize_space(stage1_pool_manifest.get("selected_personalized_method_slug"))
    CANDIDATE_RETRIEVAL_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("selected_personalized_method_label"))
else:
    raise RuntimeError(f"Unsupported CANDIDATE_POOL_ROLE: {CANDIDATE_POOL_ROLE}")
if not CANDIDATE_RETRIEVAL_METHOD_KEY or not CANDIDATE_RETRIEVAL_METHOD_LABEL:
    raise RuntimeError("Notebook 09 manifest does not contain the selected winner identity.")
SOURCE_BASELINE_METHOD_KEY = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_key"))
SOURCE_BASELINE_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_label"))
if not SOURCE_BASELINE_METHOD_KEY or not SOURCE_BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 09 manifest does not contain the query-only baseline winner identity.")
DEFAULT_CANDIDATE_POOL_TYPE = CANDIDATE_RETRIEVAL_METHOD_LABEL
DEFAULT_RETRIEVAL_METHOD = CANDIDATE_RETRIEVAL_METHOD_LABEL
DEFAULT_RETRIEVAL_METHOD_LABEL = CANDIDATE_RETRIEVAL_METHOD_LABEL
CANDIDATE_POOL_TYPE = CANDIDATE_RETRIEVAL_METHOD_LABEL
RETRIEVAL_METHOD = CANDIDATE_RETRIEVAL_METHOD_LABEL
RETRIEVAL_METHOD_LABEL = CANDIDATE_RETRIEVAL_METHOD_LABEL
CANDIDATE_SOURCE_LABEL = f"notebook09_{CANDIDATE_POOL_ROLE}_{CANDIDATE_RETRIEVAL_METHOD_KEY}"

candidate_path_text = normalize_space(output_paths_09.get(candidate_path_key))
query_cache_path_text = normalize_space(input_paths_09.get("query_cache"))
if not candidate_path_text or not query_cache_path_text:
    raise RuntimeError("Notebook 09 manifest is missing the selected candidate or query-cache path.")
CANDIDATE_POOL_PATH = Path(candidate_path_text)
QUERY_CACHE_PARQUET = Path(query_cache_path_text)
if not CANDIDATE_POOL_PATH.exists() or not QUERY_CACHE_PARQUET.exists():
    raise FileNotFoundError(f"Missing Notebook 09 dependency: {CANDIDATE_POOL_PATH}, {QUERY_CACHE_PARQUET}")

raw_candidates = pd.read_parquet(CANDIDATE_POOL_PATH)
require_columns(raw_candidates, REQUIRED_NOTEBOOK09_COLUMNS, "Notebook 09 winner pool")
if set(raw_candidates["category_id"].astype(str)) != {CATEGORY_ID}:
    raise RuntimeError("Candidate category_id mismatch.")
if set(raw_candidates["candidate_pool_role"].astype(str)) != {CANDIDATE_POOL_ROLE}:
    raise RuntimeError("Candidate role mismatch.")
if set(raw_candidates["method_slug"].astype(str)) != {CANDIDATE_RETRIEVAL_METHOD_KEY}:
    raise RuntimeError("Candidate method_slug mismatch.")
if set(raw_candidates["retrieval_method"].astype(str)) != {CANDIDATE_RETRIEVAL_METHOD_LABEL}:
    raise RuntimeError("Candidate retrieval_method mismatch.")
if not raw_candidates["target_parent_asin"].astype(str).eq(raw_candidates["gt_item_id"].astype(str)).all():
    raise RuntimeError("target_parent_asin and gt_item_id disagree.")

candidate_df = pd.DataFrame({
    "case_id": raw_candidates["case_id"].astype(str),
    "query_id": raw_candidates["query_id"].astype(str),
    "user_id": raw_candidates["user_id"].fillna("").astype(str),
    "regime": raw_candidates["regime"].map(normalize_regime),
    "sampling_bracket": raw_candidates["sampling_bracket"].fillna("").astype(str),
    "target_selection_mode": raw_candidates["target_selection_mode"].fillna("").astype(str),
    "query_method": STAGE1_QUERY_METHOD,
    "query_text": raw_candidates["query_text"].fillna("").astype(str),
    "target_item_id": raw_candidates["target_parent_asin"].astype(str),
    "gt_item_id": raw_candidates["gt_item_id"].astype(str),
    "target_timestamp_ms": to_timestamp_ms(raw_candidates["target_timestamp_ms"]).astype("int64"),
    "prior_review_n": pd.to_numeric(raw_candidates["prior_history_n"], errors="raise").astype(int),
    "prior_item_n": pd.to_numeric(raw_candidates["prior_history_n"], errors="raise").astype(int),
    "candidate_pool_role": raw_candidates["candidate_pool_role"].astype(str),
    "candidate_method_slug": raw_candidates["method_slug"].astype(str),
    "candidate_pool_type": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "retrieval_method": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "item_id": raw_candidates["candidate_parent_asin"].astype(str),
    "rank": pd.to_numeric(raw_candidates["candidate_rank"], errors="raise").astype(int),
    "score": pd.to_numeric(raw_candidates["candidate_score"], errors="raise").astype(float),
    "candidate_score_source": raw_candidates["candidate_score_source"].fillna("").astype(str),
    "is_target": as_bool_series(raw_candidates["is_gt"]),
})
if candidate_df["regime"].eq("").any():
    raise RuntimeError("Notebook 09 contains an unknown or empty regime.")
if candidate_df["query_text"].str.strip().eq("").any():
    raise RuntimeError("Notebook 09 contains empty query text.")
if not np.isfinite(candidate_df["score"].to_numpy()).all():
    raise RuntimeError("Notebook 09 contains non-finite candidate scores.")
if candidate_df.duplicated(["query_id", "item_id"]).any() or candidate_df.duplicated(["query_id", "rank"]).any():
    raise RuntimeError("Notebook 09 contains duplicate query-item or query-rank rows.")
computed_target = candidate_df["item_id"].eq(candidate_df["target_item_id"])
if not computed_target.eq(candidate_df["is_target"]).all():
    raise RuntimeError("Notebook 09 is_gt is inconsistent with candidate identity.")

manifest_budget_k = int(stage1_pool_manifest.get("candidate_budget_k", POOL_K))
expected_k = int(stage1_pool_manifest.get("effective_candidate_count_per_query", POOL_K))
if manifest_budget_k != POOL_K or expected_k != POOL_K:
    raise RuntimeError("Notebook 09 candidate depth differs from the configured depth.")
candidate_count_per_query = candidate_df.groupby("query_id").size()
rank_stats = candidate_df.groupby("query_id")["rank"].agg(["min", "max", "nunique"])
if not candidate_count_per_query.eq(expected_k).all():
    raise RuntimeError("Notebook 09 does not provide exact-K candidates for every query.")
if not ((rank_stats["min"] == 1).all() and (rank_stats["max"] == expected_k).all() and (rank_stats["nunique"] == expected_k).all()):
    raise RuntimeError("Notebook 09 candidate ranks are not exactly 1 through K.")

query_cache_schema_columns = pq.ParquetFile(QUERY_CACHE_PARQUET).schema.names
query_cache_load_columns = ["case_id"]
query_brand_leak_source_column = None
for candidate_column in [
    "brand_or_name_leak_flag",
    "query_brand_or_name_leak_flag",
    "brand_name_leak_flag",
    "query_brand_leak_flag",
]:
    if candidate_column in query_cache_schema_columns:
        query_brand_leak_source_column = candidate_column
        query_cache_load_columns.append(candidate_column)
        break
query_audit_columns = [
    "query_evidence_source",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "insufficient_review_evidence",
]
query_cache_load_columns.extend(
    column for column in query_audit_columns
    if column in query_cache_schema_columns and column not in query_cache_load_columns
)
query_cache_load_columns = list(dict.fromkeys(query_cache_load_columns))
query_cache_raw = pd.read_parquet(QUERY_CACHE_PARQUET, columns=query_cache_load_columns)
require_columns(query_cache_raw, ["case_id"], "Notebook 06 query cache")
query_cache_meta_df = query_cache_raw.copy()
query_cache_meta_df["case_id"] = query_cache_meta_df["case_id"].astype(str)
if query_cache_meta_df["case_id"].duplicated().any():
    raise RuntimeError("Notebook 06 query cache must be unique by case_id.")
active_case_ids = set(candidate_df["case_id"])
query_cache_meta_df = query_cache_meta_df[query_cache_meta_df["case_id"].isin(active_case_ids)].copy()
if set(query_cache_meta_df["case_id"]) != active_case_ids:
    raise RuntimeError("Notebook 06 query cache does not cover every Notebook 09 case.")
target_rank_lookup_frames = []

if "target_rank_desc" in raw_candidates.columns:
    target_rank_lookup_frames.append(
        raw_candidates[["case_id", "target_rank_desc"]].copy()
    )

if "target_rank_desc" in query_cache_meta_df.columns:
    target_rank_lookup_frames.append(
        query_cache_meta_df[["case_id", "target_rank_desc"]].copy()
    )

sampling_rank_paths = [
    PROJECT_ROOT / "data" / "processed" / "user_sampling" / "face_user_regime_sample.parquet",
    PROJECT_ROOT / "data" / "processed" / "user_sampling" / "face_final_sampling_pool.parquet",
]

for rank_path in sampling_rank_paths:
    if rank_path.exists():
        rank_columns = pq.ParquetFile(rank_path).schema.names
        if {"case_id", "target_rank_desc"}.issubset(rank_columns):
            target_rank_lookup_frames.append(
                pd.read_parquet(rank_path, columns=["case_id", "target_rank_desc"])
            )

if not target_rank_lookup_frames:
    raise RuntimeError("No target_rank_desc source is available.")

target_rank_lookup_df = pd.concat(target_rank_lookup_frames, ignore_index=True)
target_rank_lookup_df["case_id"] = target_rank_lookup_df["case_id"].astype(str).str.strip()
target_rank_lookup_df["target_rank_desc"] = pd.to_numeric(
    target_rank_lookup_df["target_rank_desc"],
    errors="coerce",
)
target_rank_lookup_df = (
    target_rank_lookup_df
    .dropna(subset=["case_id", "target_rank_desc"])
    .drop_duplicates("case_id", keep="first")
)

query_cache_meta_df = query_cache_meta_df.drop(columns=["target_rank_desc"], errors="ignore").merge(
    target_rank_lookup_df,
    on="case_id",
    how="left",
    validate="one_to_one",
)

if query_cache_meta_df["target_rank_desc"].isna().any():
    missing_rank_cases = query_cache_meta_df.loc[
        query_cache_meta_df["target_rank_desc"].isna(),
        ["case_id"],
    ].head(20)
    display(missing_rank_cases)
    raise RuntimeError("target_rank_desc must be available from Notebook 09, Notebook 06, or Notebook 03 sampling artifacts.")
if query_brand_leak_source_column is not None:
    query_cache_meta_df["brand_or_name_leak_flag"] = as_bool_series(query_cache_meta_df[query_brand_leak_source_column])
    query_brand_leak_qc_source = query_brand_leak_source_column
else:
    required_safe_audit_columns = [
        "query_evidence_source",
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "insufficient_review_evidence",
    ]
    missing_safe_audit_columns = [column for column in required_safe_audit_columns if column not in query_cache_meta_df.columns]
    if missing_safe_audit_columns:
        raise RuntimeError(
            "Notebook 06 query cache is missing brand leak flag columns and safe-query audit columns: "
            f"{missing_safe_audit_columns}"
        )
    if not query_cache_meta_df["query_evidence_source"].fillna("").astype(str).eq("target_review_safe_signals_only").all():
        raise RuntimeError("Cannot infer zero brand leakage unless every query uses target_review_safe_signals_only evidence.")
    unsafe_audit_flags = [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "insufficient_review_evidence",
    ]
    if any(as_bool_series(query_cache_meta_df[column]).any() for column in unsafe_audit_flags):
        raise RuntimeError("Cannot infer zero brand leakage because query audit fallback flags are active.")
    query_cache_meta_df["brand_or_name_leak_flag"] = False
    query_brand_leak_qc_source = "inferred_zero_from_safe_query_audit"
brand_terms_added_to_synthetic_query_n = int(as_bool_series(query_cache_meta_df["brand_or_name_leak_flag"]).sum())
if brand_terms_added_to_synthetic_query_n != 0:
    raise RuntimeError("Brand or item-name terms were added to a synthetic query.")
query_cache_schema_map = {
    "case_id": "case_id",
    "target_rank_desc": "notebook09_winner_pool.target_rank_desc",
    "brand_or_name_leak_flag": query_brand_leak_qc_source,
}

query_meta_df = candidate_df[[
    "case_id", "query_id", "user_id", "regime", "sampling_bracket", "target_selection_mode",
    "query_method", "query_text", "target_item_id", "gt_item_id", "target_timestamp_ms",
    "prior_review_n", "prior_item_n", "candidate_pool_type", "retrieval_method",
    "retrieval_method_label", "candidate_method_slug",
]].drop_duplicates("query_id").copy()
query_meta_df = query_meta_df.merge(
    query_cache_meta_df[["case_id", "target_rank_desc"]],
    on="case_id", how="left", validate="one_to_one",
)
query_meta_df["target_rank_desc"] = pd.to_numeric(query_meta_df["target_rank_desc"], errors="raise").astype("Int32")
query_meta_df["regime"] = pd.Categorical(query_meta_df["regime"], categories=REGIME_ORDER, ordered=True)
query_meta_df["reranking_method"] = RERANKING_METHOD

candidate_pool_type_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
retrieval_method_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
retrieval_method_label_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
EXPECTED_QUERY_COUNT = int(stage1_pool_manifest.get("query_count", query_meta_df["query_id"].nunique()))
EXPECTED_CANDIDATE_ROWS_TOP1000 = EXPECTED_QUERY_COUNT * expected_k
EXPECTED_REGIME_COUNTS = query_meta_df["regime"].astype(str).value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
if query_meta_df["query_id"].nunique() != EXPECTED_QUERY_COUNT or len(candidate_df) != EXPECTED_CANDIDATE_ROWS_TOP1000:
    raise RuntimeError("Notebook 09 manifest counts disagree with the loaded pool.")

n_queries = int(query_meta_df["query_id"].nunique())
n_candidates = int(len(candidate_df))
load_runtime_sec = time.perf_counter() - load_start
record_runtime_step("load_inputs", load_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
candidate_source_qc = {
    "stage1_candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_role": CANDIDATE_POOL_ROLE,
    "candidate_retrieval_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "candidate_retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "candidate_pool_path": str(CANDIDATE_POOL_PATH),
    "query_cache_path": str(QUERY_CACHE_PARQUET),
    "candidate_budget_policy": stage1_pool_manifest.get("candidate_budget_policy"),
    "candidate_budget_k": expected_k,
    "query_count": n_queries,
    "candidate_rows": n_candidates,
    "brand_terms_added_to_synthetic_query_n": brand_terms_added_to_synthetic_query_n,
}

print("Candidate role:", CANDIDATE_POOL_ROLE)
print("Candidate method:", CANDIDATE_RETRIEVAL_METHOD_KEY, "-", CANDIDATE_RETRIEVAL_METHOD_LABEL)
print("Candidate rows:", n_candidates, "Cases:", n_queries, "Exact K:", expected_k)
print("Synthetic-query brand leakage: PASS")


## 5. Load Candidate-Side Item Artifacts and Establish the No-Prior History Contract

In [ ]:
# =========================================================
# Exact Notebook 02 common facet contract
# =========================================================
PROFILE_ROLES = [
    "brand",
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
ATTRIBUTE_FAMILIES = tuple(PROFILE_ROLES)
FACIAL_ATTRIBUTE_FAMILIES = ATTRIBUTE_FAMILIES

if CATEGORY_ID == "face":
    FAMILY_STRENGTH_WEIGHTS = {
        "brand": 0.60,
        "category_or_product_type": 1.50,
        "form_texture": 1.00,
        "ingredient_or_composition": 1.20,
        "need_benefit_concern": 1.00,
        "claim_constraint": 0.80,
        "target_context": 0.70,
        "sensory": 0.40,
    }
else:
    FAMILY_STRENGTH_WEIGHTS = {
        "brand": 0.60,
        "category_or_product_type": 1.50,
        "form_texture": 1.10,
        "ingredient_or_composition": 1.20,
        "need_benefit_concern": 1.25,
        "claim_constraint": 0.90,
        "target_context": 0.70,
        "sensory": 0.50,
    }

SUPPLEMENTARY_FAMILY_PRIORITY = ["brand", "need_benefit_concern", "ingredient_or_composition"]
SUPPLEMENTARY_METRIC_COLS = PREFERENCE_DIAGNOSTIC_METRICS


def get_item_family_set(item_id, family):
    return item_family_values.get(str(item_id), {}).get(str(family), frozenset())


def weighted_similarity_to_target(ranked_item_ids, gt_item_id, families, family_weights=None, k=PREFERENCE_ALIGNMENT_K):
    ranked = [str(item_id) for item_id in list(ranked_item_ids)[:int(k)]]
    if family_weights is None:
        family_weights = {family: 1.0 for family in families}
    weighted_sum = 0.0
    weight_total = 0.0
    family_scores = {}
    for family in families:
        target_values = get_item_family_set(str(gt_item_id), family)
        per_item = [jaccard(get_item_family_set(item_id, family), target_values) for item_id in ranked]
        score = float(np.mean(per_item)) if per_item else np.nan
        family_scores[family] = score
        weight = float(family_weights.get(family, 0.0))
        if pd.notna(score) and weight > 0:
            weighted_sum += score * weight
            weight_total += weight
    return {
        "weighted_facet_overlap_at_5": weighted_sum / weight_total if weight_total else np.nan,
        "brand_match_at_5": family_scores.get("brand", np.nan),
        "concern_match_at_5": family_scores.get("need_benefit_concern", np.nan),
        "ingredient_match_at_5": family_scores.get("ingredient_or_composition", np.nan),
    }


print("Notebook 02 profile-safe facet roles:", PROFILE_ROLES)


In [ ]:
# The category notebooks intentionally share the Notebook 02 common-role registry.
MAIN_FAMILIES = list(ATTRIBUTE_FAMILIES)
SAFE_FACET_FALLBACK_FAMILIES = set()
ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = {}
print("Common-role Shannon registry ready.")


In [ ]:
# =========================================================
# Load candidate-side item facets and establish an empty prior-history frame.
# =========================================================
artifact_load_start = time.perf_counter()

schema_required = ["parent_asin", "brand_facet_text", "common_query_safe_facet_text"]
facet_required = ["parent_asin", "facet_role", "facet_value_norm", "is_brand", "is_review_derived", "is_profile_safe"]
item_schema_raw = pd.read_parquet(ITEM_SCHEMA_PARQUET, columns=schema_required)
items_facets_raw = pd.read_parquet(ITEMS_FACETS_PARQUET, columns=facet_required)
require_columns(item_schema_raw, schema_required, "Notebook 02 item schema")
require_columns(items_facets_raw, facet_required, "Notebook 02 common long facets")

item_schema_raw["item_id"] = item_schema_raw["parent_asin"].astype(str)
if item_schema_raw["item_id"].duplicated().any():
    raise RuntimeError("Notebook 02 item schema must be unique by parent_asin.")
item_schema_raw["brand_raw"] = item_schema_raw["brand_facet_text"].fillna("").astype(str)
item_schema_raw["brand_norm"] = item_schema_raw["brand_raw"].map(normalize_text)
item_brand_raw_map = item_schema_raw.set_index("item_id")["brand_raw"].to_dict()
item_brand_map = item_schema_raw.set_index("item_id")["brand_norm"].to_dict()

items_facets_raw["item_id"] = items_facets_raw["parent_asin"].astype(str)
items_facets_raw["facet_role"] = items_facets_raw["facet_role"].astype(str)
items_facets_raw["facet_value_norm"] = items_facets_raw["facet_value_norm"].fillna("").map(normalize_text)
unknown_roles = sorted(set(items_facets_raw["facet_role"]) - set(PROFILE_ROLES) - {"review_derived_signal"})
if unknown_roles:
    raise RuntimeError(f"Unexpected Notebook 02 facet roles: {unknown_roles}")
profile_facets = items_facets_raw[
    as_bool_series(items_facets_raw["is_profile_safe"])
    & ~as_bool_series(items_facets_raw["is_review_derived"])
    & items_facets_raw["facet_role"].isin(PROFILE_ROLES)
    & items_facets_raw["facet_value_norm"].ne("")
][["item_id", "facet_role", "facet_value_norm", "is_brand"]].drop_duplicates()
if as_bool_series(profile_facets.loc[profile_facets["facet_role"].ne("brand"), "is_brand"]).any():
    raise RuntimeError("Notebook 02 is_brand conflicts with facet_role.")

family_map = defaultdict(lambda: defaultdict(set))
for row in profile_facets.itertuples(index=False):
    family_map[str(row.item_id)][str(row.facet_role)].add(str(row.facet_value_norm))
for item_id, brand_value in item_brand_map.items():
    if brand_value:
        family_map[str(item_id)]["brand"].add(str(brand_value))
item_family_values = {
    item_id: {role: frozenset(values) for role, values in role_map.items() if values}
    for item_id, role_map in family_map.items()
}

candidate_item_ids = set(candidate_df["item_id"].astype(str))
missing_candidate_items = sorted(candidate_item_ids - set(item_brand_map))
if missing_candidate_items:
    raise RuntimeError(f"Notebook 02 item schema misses candidate items: n={len(missing_candidate_items)}")
candidate_df["candidate_brand_raw"] = candidate_df["item_id"].map(item_brand_raw_map).fillna("").astype(str)
candidate_df["candidate_brand_norm"] = candidate_df["item_id"].map(item_brand_map).fillna("").astype(str)
candidate_df["candidate_brand_present"] = candidate_df["candidate_brand_norm"].ne("").astype("int8")
candidate_brand_nonnull_rate = float(candidate_df["candidate_brand_present"].mean())

prior_history_schema_map = {}
prior_history_df = pd.DataFrame(columns=[
    "case_id", "query_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms",
])
same_target_item_prior_rows = 0
temporal_validation_passed = True
if USE_USER_PRIOR_FEATURES:
    if CATEGORY_ID == "face":
        prior_required = ["case_id", "user_id", "target_timestamp_ms", "prior_item_id", "prior_timestamp_ms"]
        prior_item_column = "prior_item_id"
    else:
        prior_required = ["case_id", "user_id", "target_parent_asin", "target_timestamp_ms", "prior_parent_asin", "prior_timestamp_ms"]
        prior_item_column = "prior_parent_asin"
    prior_raw = pd.read_parquet(PRIOR_HISTORY_PARQUET, columns=prior_required)
    require_columns(prior_raw, prior_required, "Notebook 03 All Prior")
    prior_history_schema_map = {column: column for column in prior_required}
    prior_std = pd.DataFrame({
        "case_id": prior_raw["case_id"].astype(str),
        "user_id": prior_raw["user_id"].fillna("").astype(str),
        "prior_item_id": prior_raw[prior_item_column].fillna("").astype(str),
        "prior_timestamp_ms": to_timestamp_ms(prior_raw["prior_timestamp_ms"]),
        "upstream_target_timestamp_ms": to_timestamp_ms(prior_raw["target_timestamp_ms"]),
    })
    query_lookup = query_meta_df[["case_id", "query_id", "user_id", "target_item_id", "target_timestamp_ms"]].copy()
    prior_std = prior_std.merge(query_lookup, on=["case_id", "user_id"], how="inner", validate="many_to_one")
    if not prior_std["upstream_target_timestamp_ms"].astype("int64").eq(prior_std["target_timestamp_ms"].astype("int64")).all():
        raise RuntimeError("Notebook 03 and Notebook 09 target timestamps disagree.")
    prior_std = prior_std[
        prior_std["user_id"].ne("") & prior_std["prior_item_id"].ne("") & prior_std["prior_timestamp_ms"].notna()
    ].copy()
    prior_std["prior_timestamp_ms"] = prior_std["prior_timestamp_ms"].astype("int64")
    temporal_validation_passed = bool(prior_std["prior_timestamp_ms"].lt(prior_std["target_timestamp_ms"]).all())
    if not temporal_validation_passed:
        raise RuntimeError("All Prior contains an interaction at or after the target timestamp.")
    same_target_item_prior_rows = int(prior_std["prior_item_id"].eq(prior_std["target_item_id"]).sum())
    if same_target_item_prior_rows != 0:
        raise RuntimeError("All Prior contains the held-out target parent item.")
    prior_history_df = prior_std[[
        "case_id", "query_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms",
    ]].reset_index(drop=True)
    missing_prior_items = sorted(set(prior_history_df["prior_item_id"]) - set(item_brand_map))
    if missing_prior_items:
        raise RuntimeError(f"Notebook 02 item schema misses All Prior items: n={len(missing_prior_items)}")
    actual_prior_count = prior_history_df.groupby("query_id").size().reindex(query_meta_df["query_id"], fill_value=0).astype(int)
    cold_by_query = query_meta_df.set_index("query_id")["regime"].astype(str).eq("cold")
    if (cold_by_query != actual_prior_count.eq(0)).any():
        raise RuntimeError("Cold regime disagrees with the exact All Prior event count.")

global_value_vocab = {family: set() for family in ATTRIBUTE_FAMILIES}
family_item_support = {family: 0 for family in ATTRIBUTE_FAMILIES}
for family_values in item_family_values.values():
    for family in ATTRIBUTE_FAMILIES:
        values = set(family_values.get(family, frozenset()))
        if values:
            family_item_support[family] += 1
            global_value_vocab[family].update(values)
raw_global_family_weight = {
    family: float(family_item_support[family]) * float(FAMILY_STRENGTH_WEIGHTS[family])
    for family in ATTRIBUTE_FAMILIES
}
global_weight_total = float(sum(raw_global_family_weight.values()))
global_family_weights = {
    family: raw_global_family_weight[family] / global_weight_total if global_weight_total else 0.0
    for family in ATTRIBUTE_FAMILIES
}
ITEM_FACET_SOURCE_QC = {
    "policy": ITEM_FACET_EVIDENCE_POLICY,
    "rows_total": int(len(items_facets_raw)),
    "rows_retained": int(len(profile_facets)),
    "review_derived_rows_excluded": int(as_bool_series(items_facets_raw["is_review_derived"]).sum()),
    "filter_source": "exact_notebook02_is_profile_safe_and_not_review_derived",
    "brand_column": "brand_facet_text",
}

artifact_load_runtime_sec = time.perf_counter() - artifact_load_start
record_runtime_step("load_item_and_history_artifacts", artifact_load_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
print("Candidate brand non-null rate:", round(candidate_brand_nonnull_rate, 6))
print("All Prior events:", len(prior_history_df), "Temporal validation:", temporal_validation_passed)
print("Notebook 04 artifacts read directly:", False)


## 6. Build the No-Prior Shannon Feature Contract

In [ ]:
# =========================================================
# Build no-prior Shannon profile and candidate features.
# =========================================================
def normalized_entropy(counter_obj, vocab_size):
    total = float(sum(counter_obj.values()))
    if total <= 0:
        return np.nan
    probabilities = np.asarray([value / total for value in counter_obj.values()], dtype=np.float64)
    entropy = float(-np.sum(probabilities * np.log(probabilities + 1e-12)))
    if vocab_size <= 1:
        return 0.0
    return float(np.clip(entropy / math.log(vocab_size), 0.0, 1.0))


FUNCTIONAL_FAMILIES = tuple(family for family in ATTRIBUTE_FAMILIES if family != "brand")
if not FUNCTIONAL_FAMILIES:
    raise RuntimeError("Shannon requires at least one non-brand functional family.")


def normalize_weight_dict(raw_dict):
    cleaned = {family: max(float(raw_dict.get(family, 0.0)), 0.0) for family in ATTRIBUTE_FAMILIES}
    total = float(sum(cleaned.values()))
    return {family: cleaned[family] / total if total else 0.0 for family in ATTRIBUTE_FAMILIES}


def history_factor(history_n):
    history_n = max(int(history_n), 0)
    return float(min(1.0, math.log1p(history_n) / math.log1p(SHANNON_HISTORY_SATURATION_N))) if history_n else 0.0


def compute_personalization_strength(regime, entropy_norm, history_n):
    if not USE_USER_PRIOR_FEATURES:
        return 0.0
    regime = normalize_regime(regime)
    if regime == "cold" or history_n <= 0 or pd.isna(entropy_norm):
        return 0.0
    concentration = float(np.clip(1.0 - float(entropy_norm), 0.0, 1.0))
    raw_strength = SHANNON_ALPHA * concentration * history_factor(history_n)
    return float(np.clip(raw_strength, 0.0, float(REGIME_STRENGTH_CAP[regime])))


def build_query_profiles(query_meta, prior_history):
    prior_items_by_query = prior_history.groupby("query_id", sort=False)["prior_item_id"].apply(list).to_dict() if len(prior_history) else {}
    rows = []
    profile_value_sets = {}
    profile_counters = {}
    for row in query_meta.itertuples(index=False):
        query_id = str(row.query_id)
        prior_items = [str(item_id) for item_id in prior_items_by_query.get(query_id, [])]
        unique_prior_items = set(prior_items)
        family_counters = {family: Counter() for family in ATTRIBUTE_FAMILIES}
        if USE_USER_PRIOR_FEATURES:
            # All Prior frequency is intentionally retained: repeated interactions are not deduplicated.
            for item_id in prior_items:
                for family in ATTRIBUTE_FAMILIES:
                    for value in get_item_family_set(item_id, family):
                        family_counters[family][value] += 1
        family_entropy = {
            family: normalized_entropy(family_counters[family], max(len(global_value_vocab[family]), 1))
            for family in ATTRIBUTE_FAMILIES
        }
        raw_weights = {
            family: math.log1p(sum(family_counters[family].values())) * float(FAMILY_STRENGTH_WEIGHTS[family])
            for family in ATTRIBUTE_FAMILIES
        }
        family_weights = normalize_weight_dict(raw_weights)
        # Keep brand outside the functional Shannon entropy gate. Brand is scored in its own compact block.
        entropy_terms = [family_entropy[family] for family in FUNCTIONAL_FAMILIES if pd.notna(family_entropy[family]) and family_weights[family] > 0]
        entropy_weights = [family_weights[family] for family in FUNCTIONAL_FAMILIES if pd.notna(family_entropy[family]) and family_weights[family] > 0]
        entropy_norm_profile = float(np.average(entropy_terms, weights=entropy_weights)) if entropy_terms else np.nan
        strength = compute_personalization_strength(str(row.regime), entropy_norm_profile, len(prior_items))
        value_sets = {family: frozenset(family_counters[family]) for family in ATTRIBUTE_FAMILIES}
        profile_value_sets[query_id] = value_sets
        profile_counters[query_id] = family_counters
        out_row = {
            "case_id": str(row.case_id),
            "query_id": query_id,
            "user_id": str(row.user_id),
            "regime": str(row.regime),
            "profile_history_review_n": int(len(prior_items)),
            "profile_history_item_n": int(len(unique_prior_items)),
            "user_entropy_norm_shannon": entropy_norm_profile,
            "profile_concentration": float(1.0 - entropy_norm_profile) if pd.notna(entropy_norm_profile) else np.nan,
            "history_factor": history_factor(len(prior_items)),
            "lambda_shannon": strength,
            "profile_values_json": json.dumps({family: sorted(value_sets[family]) for family in ATTRIBUTE_FAMILIES}, ensure_ascii=False),
        }
        if USE_USER_PRIOR_FEATURES:
            for family in ATTRIBUTE_FAMILIES:
                out_row[f"profile_weight__{family}"] = family_weights[family]
                out_row[f"profile_value_count__{family}"] = int(len(value_sets[family]))
                out_row[f"profile_entropy_norm__{family}"] = family_entropy[family]
        rows.append(out_row)
    return pd.DataFrame(rows), profile_value_sets, profile_counters


feature_start = time.perf_counter()
query_profiles_df, profile_value_sets_by_query, profile_counters_by_query = build_query_profiles(query_meta_df, prior_history_df)
candidate_identity_before = candidate_df[["query_id", "item_id"]].sort_values(["query_id", "item_id"]).reset_index(drop=True)
feature_base_df = candidate_df.merge(
    query_profiles_df.drop(columns=["user_id", "regime"], errors="ignore"),
    on=["case_id", "query_id"], how="left", validate="many_to_one",
)
candidate_identity_after = feature_base_df[["query_id", "item_id"]].sort_values(["query_id", "item_id"]).reset_index(drop=True)
if len(feature_base_df) != len(candidate_df) or not candidate_identity_before.equals(candidate_identity_after):
    raise RuntimeError("Feature construction changed candidate rows or candidate IDs.")

feature_base_df["profile_history_review_n"] = safe_int_series(feature_base_df["profile_history_review_n"], default=0)
feature_base_df["profile_history_item_n"] = safe_int_series(feature_base_df["profile_history_item_n"], default=0)
feature_base_df["lambda_shannon"] = safe_numeric(feature_base_df["lambda_shannon"], default=0.0).clip(0.0, 1.0).astype("float32")
feature_base_df["retrieval_score_norm_full_pool"] = minmax_by_group(feature_base_df, "query_id", "score", max_rank=POOL_K).astype("float32")

functional_personalization_score = np.zeros(len(feature_base_df), dtype=np.float32)
if USE_USER_PRIOR_FEATURES:
    query_ids = feature_base_df["query_id"].astype(str).to_numpy()
    item_ids = feature_base_df["item_id"].astype(str).to_numpy()
    profiles_by_query = query_profiles_df.set_index("query_id")
    for family in ATTRIBUTE_FAMILIES:
        weights_by_query = profiles_by_query[f"profile_weight__{family}"].to_dict()
        matches = np.fromiter(
            (jaccard(profile_value_sets_by_query[qid][family], get_item_family_set(item_id, family)) for qid, item_id in zip(query_ids, item_ids)),
            dtype=np.float32, count=len(feature_base_df),
        )
        weights = np.fromiter((float(weights_by_query[qid]) for qid in query_ids), dtype=np.float32, count=len(feature_base_df))
        feature_base_df[f"match__{family}"] = matches.astype("float32")
        feature_base_df[f"weight__{family}"] = weights.astype("float32")
        if family != "brand":
            functional_personalization_score += matches * weights
feature_base_df["functional_personalization_score"] = np.clip(
    functional_personalization_score, 0.0, 1.0
).astype("float32")
feature_base_df["brand_personalization_score"] = np.float32(0.0)
feature_base_df["personalization_score"] = feature_base_df["functional_personalization_score"]

USER_BRAND_AFFINITY_COLUMNS = [
    "candidate_brand_prior_interaction_count",
    "candidate_brand_prior_share",
    "candidate_brand_recency_weight",
    "user_prior_brand_entropy_norm",
]
if USE_USER_PRIOR_FEATURES:
    brand_events = prior_history_df.copy()
    brand_events["brand"] = brand_events["prior_item_id"].map(item_brand_map).fillna("").astype(str)
    brand_events = brand_events[brand_events["brand"].ne("")].copy()
    interaction_count = brand_events.groupby(["query_id", "brand"]).size().astype(int).to_dict()
    unique_item_count = brand_events.groupby(["query_id", "brand"])["prior_item_id"].nunique().astype(int).to_dict()
    brand_event_denominator = brand_events.groupby("query_id").size().astype(int).to_dict()
    brand_last_timestamp = brand_events.groupby(["query_id", "brand"])["prior_timestamp_ms"].max().astype("int64").to_dict()
    unique_brand_count = brand_events.groupby("query_id")["brand"].nunique().astype(int).to_dict()
    brand_entropy_norm = {}
    for query_id, group in brand_events.groupby("query_id", sort=False):
        entropy_value = normalized_entropy(
            Counter(group["brand"].astype(str)),
            max(len(global_value_vocab.get("brand", frozenset())), 1),
        )
        brand_entropy_norm[str(query_id)] = 0.0 if pd.isna(entropy_value) else float(entropy_value)
    dominant_brand = {}
    for query_id, group in brand_events.groupby("query_id", sort=False):
        counts = group["brand"].value_counts()
        max_count = int(counts.max())
        dominant_brand[str(query_id)] = sorted(counts[counts.eq(max_count)].index.astype(str))[0]
    keys = list(zip(feature_base_df["query_id"].astype(str), feature_base_df["candidate_brand_norm"].astype(str)))
    feature_base_df["candidate_brand_seen_in_prior"] = np.asarray([interaction_count.get(key, 0) > 0 for key in keys], dtype=np.int8)
    feature_base_df["candidate_brand_prior_interaction_count"] = np.asarray([interaction_count.get(key, 0) for key in keys], dtype=np.int32)
    feature_base_df["candidate_brand_prior_unique_item_count"] = np.asarray([unique_item_count.get(key, 0) for key in keys], dtype=np.int32)
    feature_base_df["candidate_brand_prior_share"] = np.asarray([
        interaction_count.get(key, 0) / max(brand_event_denominator.get(key[0], 0), 1) for key in keys
    ], dtype=np.float32)
    recency_values = []
    for key, target_timestamp in zip(keys, feature_base_df["target_timestamp_ms"]):
        last_timestamp = brand_last_timestamp.get(key)
        age_days = (int(target_timestamp) - int(last_timestamp)) / float(MS_PER_DAY) if last_timestamp is not None else np.inf
        recency_values.append(math.exp(-max(age_days, 0.0) / 180.0) if np.isfinite(age_days) else 0.0)
    feature_base_df["candidate_brand_recency_weight"] = np.asarray(recency_values, dtype=np.float32)
    feature_base_df["candidate_brand_is_dominant_prior_brand"] = np.asarray([
        bool(key[1]) and dominant_brand.get(key[0], "") == key[1] for key in keys
    ], dtype=np.int8)
    feature_base_df["user_prior_unique_brand_count"] = feature_base_df["query_id"].map(unique_brand_count).fillna(0).astype("int32")
    feature_base_df["user_prior_brand_entropy_norm"] = (
        feature_base_df["query_id"].map(brand_entropy_norm).fillna(0.0).clip(0.0, 1.0).astype("float32")
    )
    feature_base_df["candidate_brand_profile_weight"] = (
        feature_base_df["match__brand"] * feature_base_df["weight__brand"]
    ).astype("float32")
    brand_interaction_strength = (
        np.log1p(feature_base_df["candidate_brand_prior_interaction_count"].clip(lower=0))
        / np.log1p(feature_base_df["profile_history_review_n"].clip(lower=1))
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).clip(0.0, 1.0)
    brand_loyalty = (1.0 - feature_base_df["user_prior_brand_entropy_norm"]).clip(0.0, 1.0)
    brand_component_raw = (
        (brand_interaction_strength
         + feature_base_df["candidate_brand_prior_share"].clip(0.0, 1.0)
         + feature_base_df["candidate_brand_recency_weight"].clip(0.0, 1.0)
         + brand_loyalty)
        / 4.0
    ) * feature_base_df["candidate_brand_seen_in_prior"].astype("float32")
    feature_base_df["brand_personalization_score"] = (
        brand_component_raw * feature_base_df["weight__brand"]
    ).clip(0.0, 1.0).astype("float32")
    feature_base_df["personalization_score"] = (
        feature_base_df["functional_personalization_score"]
        + feature_base_df["brand_personalization_score"]
    ).clip(0.0, 1.0).astype("float32")
    users_with_prior_brand = int(brand_events["query_id"].nunique())
    candidate_brand_prior_match_rate = float(feature_base_df["candidate_brand_seen_in_prior"].mean())
else:
    users_with_prior_brand = 0
    candidate_brand_prior_match_rate = 0.0

feature_base_df["shannon_feature_available"] = (
    feature_base_df["profile_history_review_n"].gt(0)
    & feature_base_df["personalization_score"].gt(0)
)

# The no-prior control is an exact Stage-1 no-op; candidate brand presence is not a scoring feature.
S2Q_FEATURE_COLUMNS = []
SHARED_ALL_PRIOR_FEATURE_COLUMNS = [
    "candidate_brand_present",
    *USER_BRAND_AFFINITY_COLUMNS,
    "lambda_shannon",
    "functional_personalization_score",
    "brand_personalization_score",
]
SHANNON_FEATURE_COLUMNS = SHARED_ALL_PRIOR_FEATURE_COLUMNS if USE_USER_PRIOR_FEATURES else S2Q_FEATURE_COLUMNS
missing_feature_columns = [column for column in SHANNON_FEATURE_COLUMNS if column not in feature_base_df.columns]
if missing_feature_columns:
    raise RuntimeError(f"Missing Shannon contract features: {missing_feature_columns}")
if len(SHANNON_FEATURE_COLUMNS) != len(set(SHANNON_FEATURE_COLUMNS)):
    raise RuntimeError("Duplicated Shannon feature columns.")
user_brand_feature_columns_S2Q = sorted(set(S2Q_FEATURE_COLUMNS) & set(USER_BRAND_AFFINITY_COLUMNS))
if user_brand_feature_columns_S2Q:
    raise RuntimeError("S2-Q includes user-brand affinity columns.")

feature_runtime_sec = time.perf_counter() - feature_start
record_runtime_step("feature_preparation", feature_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
record_runtime_step("model_fit_or_tuning", 0.0, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates, note="deterministic heuristic; no training or tuning")
global_family_weights_df = pd.DataFrame([{
    "family": family,
    "family_strength_weight": FAMILY_STRENGTH_WEIGHTS[family],
    "item_support": family_item_support[family],
    "vocab_size": len(global_value_vocab[family]),
    "global_family_weight": global_family_weights[family],
    "mean_profile_weight": float(query_profiles_df[f"profile_weight__{family}"].mean()) if USE_USER_PRIOR_FEATURES else 0.0,
    "mean_match": float(feature_base_df[f"match__{family}"].mean()) if USE_USER_PRIOR_FEATURES else 0.0,
} for family in ATTRIBUTE_FAMILIES])

print("Candidate rows unchanged by feature construction: PASS")
print("Users with at least one prior brand:", users_with_prior_brand)
print("Candidate-brand prior-match rate:", round(candidate_brand_prior_match_rate, 6))


In [ ]:
# =========================================================
# Candidate-Side Temporal Diagnostics and No-Prior Shannon Contract
# =========================================================
def _load_json_local(path_obj):
    path_obj = Path(path_obj)
    if not path_obj.exists():
        return {}
    with open(path_obj, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def _days_between_ms(later_ms, earlier_ms):
    return (np.asarray(later_ms, dtype=np.float64) - np.asarray(earlier_ms, dtype=np.float64)) / float(MS_PER_DAY)


def build_item_timestamp_map(item_time_df):
    if item_time_df.empty:
        return {}
    work = item_time_df[["parent_asin", "review_timestamp_ms"]].copy()
    work["parent_asin"] = work["parent_asin"].fillna("").astype(str).str.strip()
    work["review_timestamp_ms"] = pd.to_numeric(work["review_timestamp_ms"], errors="coerce")
    work = work.dropna(subset=["review_timestamp_ms"])
    work = work[work["parent_asin"].ne("")]
    work["review_timestamp_ms"] = work["review_timestamp_ms"].astype(np.int64)
    out = {}
    for item_id, group in work.groupby("parent_asin", sort=False):
        out[str(item_id)] = np.sort(group["review_timestamp_ms"].to_numpy(dtype=np.int64))
    return out


def add_shannon_item_temporal_features(df, item_ts_map):
    n = len(df)
    prequery_count = np.zeros(n, dtype=np.float32)
    last_gap_days = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    recent_90d = np.zeros(n, dtype=np.float32)
    item_values = df["item_id"].astype(str).to_numpy()
    qts_values = pd.to_numeric(df["target_timestamp_ms"], errors="coerce").fillna(0).astype(np.int64).to_numpy()
    for idx, (item_id, qts) in enumerate(zip(item_values, qts_values)):
        if qts <= 0:
            continue
        ts = item_ts_map.get(str(item_id))
        if ts is None or len(ts) == 0:
            continue
        right = np.searchsorted(ts, qts, side="left")
        prequery_count[idx] = float(right)
        if right > 0:
            last_gap_days[idx] = max(0.0, float(qts - int(ts[right - 1])) / float(MS_PER_DAY))
        left_90 = np.searchsorted(ts, qts - int(90 * MS_PER_DAY), side="left")
        recent_90d[idx] = float(max(0, right - left_90))
    df["item_prequery_review_count"] = prequery_count
    df["item_recent_review_count_90d"] = recent_90d
    df["item_recent_review_share_90d"] = (recent_90d / np.maximum(prequery_count, 1.0)).astype(np.float32)
    df["item_last_review_gap_days"] = last_gap_days
    df["item_review_velocity_90d"] = (recent_90d / np.float32(90.0)).astype(np.float32)
    return df


def build_case_user_temporal_maps(prior_history_source):
    if prior_history_source.empty:
        return {"user_last_ts": {}, "user_item_last_ts": {}, "user_item_recent_count_180d": {}, "user_facet_last_ts": {}, "future_or_equal_excluded_n": 0}
    required = ["case_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms"]
    missing = [c for c in required if c not in prior_history_source.columns]
    if missing:
        raise RuntimeError(f"prior_history_df is missing required temporal columns: {missing}")
    prior = prior_history_source[required].copy()
    prior["case_id"] = prior["case_id"].fillna("").astype(str).str.strip()
    prior["user_id"] = prior["user_id"].fillna("").astype(str).str.strip()
    prior["prior_item_id"] = prior["prior_item_id"].fillna("").astype(str).str.strip()
    prior["prior_timestamp_ms"] = pd.to_numeric(prior["prior_timestamp_ms"], errors="coerce")
    prior["target_timestamp_ms"] = pd.to_numeric(prior["target_timestamp_ms"], errors="coerce")
    prior = prior.dropna(subset=["prior_timestamp_ms", "target_timestamp_ms"])
    before_n = len(prior)
    prior = prior[prior["prior_timestamp_ms"].lt(prior["target_timestamp_ms"])].copy()
    future_or_equal_excluded_n = int(before_n - len(prior))
    if prior.empty:
        return {"user_last_ts": {}, "user_item_last_ts": {}, "user_item_recent_count_180d": {}, "user_facet_last_ts": {}, "future_or_equal_excluded_n": future_or_equal_excluded_n}
    prior["prior_timestamp_ms"] = prior["prior_timestamp_ms"].astype(np.int64)
    prior["target_timestamp_ms"] = prior["target_timestamp_ms"].astype(np.int64)
    prior["event_age_days"] = (prior["target_timestamp_ms"] - prior["prior_timestamp_ms"]) / float(MS_PER_DAY)
    user_last_ts = prior.groupby(["case_id", "user_id"], sort=False)["prior_timestamp_ms"].max().to_dict()
    user_item_last_ts = prior.groupby(["case_id", "user_id", "prior_item_id"], sort=False)["prior_timestamp_ms"].max().to_dict()
    user_item_recent_count_180d = (
        prior.loc[prior["event_age_days"].le(180.0)]
        .groupby(["case_id", "user_id", "prior_item_id"], sort=False)
        .size()
        .astype(int)
        .to_dict()
    )
    facet_rows = []
    for item_id, family_map in item_family_values.items():
        for family in FACIAL_ATTRIBUTE_FAMILIES:
            for label in family_map.get(family, frozenset()):
                if label:
                    facet_rows.append({"item_id": str(item_id), "family": str(family), "label": str(label)})
    if facet_rows:
        facet_key = pd.DataFrame(facet_rows).drop_duplicates()
        prior_facet = prior.merge(facet_key, left_on="prior_item_id", right_on="item_id", how="inner")
        user_facet_last_ts = prior_facet.groupby(["case_id", "user_id", "family", "label"], sort=False)["prior_timestamp_ms"].max().to_dict() if len(prior_facet) else {}
    else:
        user_facet_last_ts = {}
    return {
        "user_last_ts": user_last_ts,
        "user_item_last_ts": user_item_last_ts,
        "user_item_recent_count_180d": user_item_recent_count_180d,
        "user_facet_last_ts": user_facet_last_ts,
        "future_or_equal_excluded_n": future_or_equal_excluded_n,
    }


def add_shannon_user_temporal_features(df, temporal_maps):
    n = len(df)
    user_gap = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_gap = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_recent = np.zeros(n, dtype=np.float32)
    family_gap_cols = {family: np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32) for family in FACIAL_ATTRIBUTE_FAMILIES}
    user_last_ts = temporal_maps.get("user_last_ts", {})
    user_item_last_ts = temporal_maps.get("user_item_last_ts", {})
    user_item_recent_count = temporal_maps.get("user_item_recent_count_180d", {})
    user_facet_last_ts = temporal_maps.get("user_facet_last_ts", {})
    for idx, row in enumerate(df[["case_id", "user_id", "item_id", "target_timestamp_ms"]].itertuples(index=False)):
        case_id = str(row.case_id)
        user_id = str(row.user_id)
        item_id = str(row.item_id)
        qts = int(row.target_timestamp_ms) if pd.notna(row.target_timestamp_ms) else 0
        if qts <= 0:
            continue
        last_ts = user_last_ts.get((case_id, user_id))
        if last_ts is not None and int(last_ts) < qts:
            user_gap[idx] = max(0.0, float(qts - int(last_ts)) / float(MS_PER_DAY))
        item_key = (case_id, user_id, item_id)
        last_item_ts = user_item_last_ts.get(item_key)
        if last_item_ts is not None and int(last_item_ts) < qts:
            user_item_gap[idx] = max(0.0, float(qts - int(last_item_ts)) / float(MS_PER_DAY))
        user_item_recent[idx] = float(user_item_recent_count.get(item_key, 0))
        item_family_map = item_family_values.get(item_id, {})
        for family in FACIAL_ATTRIBUTE_FAMILIES:
            labels = [label for label in item_family_map.get(family, frozenset()) if label]
            ts_values = [int(user_facet_last_ts[(case_id, user_id, family, label)]) for label in labels if (case_id, user_id, family, label) in user_facet_last_ts and int(user_facet_last_ts[(case_id, user_id, family, label)]) < qts]
            if ts_values:
                family_gap_cols[family][idx] = max(0.0, float(qts - max(ts_values)) / float(MS_PER_DAY))
    df["user_last_interaction_gap_days"] = user_gap
    df["user_item_recency_days"] = user_item_gap
    df["user_item_recent_count_180d"] = user_item_recent
    for family in FACIAL_ATTRIBUTE_FAMILIES:
        df[f"user_{family}_recency_days"] = family_gap_cols[family]
    return df


def scale_recency_score(df):
    item_recent = safe_numeric(df.get("item_recent_review_share_90d", 0.0), default=0.0).clip(0.0, 1.0)
    item_gap = safe_numeric(df.get("item_last_review_gap_days", RECENCY_FEATURE_FILL_DAYS), default=RECENCY_FEATURE_FILL_DAYS)
    user_item_gap = safe_numeric(df.get("user_item_recency_days", RECENCY_FEATURE_FILL_DAYS), default=RECENCY_FEATURE_FILL_DAYS)
    user_gap = safe_numeric(df.get("user_last_interaction_gap_days", RECENCY_FEATURE_FILL_DAYS), default=RECENCY_FEATURE_FILL_DAYS)
    item_gap_score = np.exp(-np.minimum(item_gap, RECENCY_FEATURE_FILL_DAYS) / 180.0)
    user_item_score = np.exp(-np.minimum(user_item_gap, RECENCY_FEATURE_FILL_DAYS) / 180.0)
    user_gap_score = np.exp(-np.minimum(user_gap, RECENCY_FEATURE_FILL_DAYS) / 365.0)
    score = 0.40 * item_recent + 0.25 * item_gap_score + 0.25 * user_item_score + 0.10 * user_gap_score
    return pd.Series(np.clip(score, 0.0, 1.0), index=df.index, dtype="float32")

missing_temporal_paths = {}
temporal_artifacts_available = False

if USE_TEMPORAL_RECENCY_FEATURES:
    missing_temporal_paths = {
        name: str(path)
        for name, path in TEMPORAL_ARTIFACT_PATHS.items()
        if name != "temporal_artifact_manifest" and not Path(path).exists()
    }
    temporal_artifacts_available = not bool(missing_temporal_paths)

    if temporal_artifacts_available:
        item_review_time_index = pd.read_parquet(ITEM_REVIEW_TIME_INDEX_PATH)
        item_review_daily_counts = pd.read_parquet(ITEM_REVIEW_DAILY_COUNTS_PATH)
        entity_review_daily_counts = pd.read_parquet(ENTITY_REVIEW_DAILY_COUNTS_PATH)
        item_temporal_summary = pd.read_parquet(ITEM_TEMPORAL_SUMMARY_PATH)
        temporal_artifact_manifest = _load_json_local(TEMPORAL_ARTIFACT_MANIFEST_PATH)
    else:
        print("Temporal artifact files are missing; using prior-history-only temporal recency features.")
        print(json.dumps(missing_temporal_paths, indent=2))
        item_review_time_index = pd.DataFrame(columns=["parent_asin", "review_timestamp_ms"])
        item_review_daily_counts = pd.DataFrame()
        entity_review_daily_counts = pd.DataFrame()
        item_temporal_summary = pd.DataFrame()
        temporal_artifact_manifest = {
            "temporal_artifacts_available": False,
            "missing_temporal_paths": missing_temporal_paths,
            "fallback_policy": "prior_history_only_temporal_recency_features",
        }
else:
    item_review_time_index = pd.DataFrame(columns=["parent_asin", "review_timestamp_ms"])
    item_review_daily_counts = pd.DataFrame()
    entity_review_daily_counts = pd.DataFrame()
    item_temporal_summary = pd.DataFrame()
    temporal_artifact_manifest = {
        "temporal_artifacts_available": False,
        "fallback_policy": "temporal_recency_features_disabled",
    }

item_timestamp_map = build_item_timestamp_map(item_review_time_index)
case_user_temporal_maps = {"future_or_equal_excluded_n": 0}
feature_base_df = add_shannon_item_temporal_features(feature_base_df, item_timestamp_map)
# User-history temporal features are disabled in the Base no-prior control.
feature_base_df["shannon_temporal_recency_score"] = np.float32(0.0)

temporal_feature_cols = [
    "item_prequery_review_count",
    "item_recent_review_count_90d",
    "item_recent_review_share_90d",
    "item_review_velocity_90d",
    "item_last_review_gap_days",
]
temporal_feature_cols = [col for col in temporal_feature_cols if col in feature_base_df.columns]

temporal_feature_summary_df = pd.DataFrame({
    "feature": temporal_feature_cols,
    "nonnull_rate": [float(feature_base_df[col].notna().mean()) for col in temporal_feature_cols],
    "mean": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).mean()) for col in temporal_feature_cols],
    "min": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).min()) for col in temporal_feature_cols],
    "max": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).max()) for col in temporal_feature_cols],
})

diagnostic_feature_cols = list(dict.fromkeys(
    temporal_feature_cols
    + [
        "profile_history_review_n",
        "profile_history_item_n",
        "user_entropy_norm_shannon",
        "profile_concentration",
        "history_factor",
        "lambda_shannon",
        "personalization_score",
        "shannon_temporal_recency_score",
    ]
))
diagnostic_feature_cols = [col for col in diagnostic_feature_cols if col in feature_base_df.columns or col in query_profiles_df.columns]

used_model_features_df = pd.DataFrame({
    "feature": diagnostic_feature_cols,
    "feature_group": [
        "item_temporal_diagnostic" if col in temporal_feature_cols else "prior_profile_diagnostic"
        for col in diagnostic_feature_cols
    ],
    "used_for_final_score": [False for _ in diagnostic_feature_cols],
    "is_score_feature": [False for _ in diagnostic_feature_cols],
    "is_diagnostic_only": [True for _ in diagnostic_feature_cols],
    "is_common_model_feature": [False for _ in diagnostic_feature_cols],
    "is_temporal_feature": [col in temporal_feature_cols or col == "shannon_temporal_recency_score" for col in diagnostic_feature_cols],
    "is_dynamic_pool_feature": [False for _ in diagnostic_feature_cols],
    "passes_no_prior_policy": [True for _ in diagnostic_feature_cols],
})

lambda_shannon_disabled = bool(
    "lambda_shannon" in query_profiles_df.columns
    and pd.to_numeric(query_profiles_df["lambda_shannon"], errors="coerce").fillna(0.0).eq(0.0).all()
)
personalization_score_disabled = bool(
    "personalization_score" in feature_base_df.columns
    and pd.to_numeric(feature_base_df["personalization_score"], errors="coerce").fillna(0.0).eq(0.0).all()
)
shannon_temporal_score_disabled = bool(
    "shannon_temporal_recency_score" in feature_base_df.columns
    and pd.to_numeric(feature_base_df["shannon_temporal_recency_score"], errors="coerce").fillna(0.0).eq(0.0).all()
)
no_prior_model_feature_contract_clean = bool(
    float(SHANNON_ALPHA) == 0.0
    and float(SHANNON_TEMPORAL_RECENCY_WEIGHT) == 0.0
    and lambda_shannon_disabled
    and personalization_score_disabled
    and shannon_temporal_score_disabled
)
no_prior_score_components_disabled = bool(
    globals().get("NO_PRIOR_ABLATION", False)
    and no_prior_model_feature_contract_clean
)
if not no_prior_score_components_disabled:
    raise RuntimeError(
        "No-prior Shannon score components are not fully disabled: "
        + json.dumps({
            "SHANNON_ALPHA": float(SHANNON_ALPHA),
            "SHANNON_TEMPORAL_RECENCY_WEIGHT": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
            "lambda_shannon_disabled": lambda_shannon_disabled,
            "personalization_score_disabled": personalization_score_disabled,
            "shannon_temporal_score_disabled": shannon_temporal_score_disabled,
        })
    )

feature_leakage_qc_df = pd.DataFrame([
    {
        "check_name": "raw_timestamp_columns_excluded",
        "passed": True,
        "detail": "Shannon is deterministic; raw timestamps are metadata only and are not used as score features.",
    },
    {
        "check_name": "target_identifier_columns_excluded",
        "passed": True,
        "detail": "Identifiers are used only for joins, grouping, and evaluation.",
    },
    {
        "check_name": "labels_excluded_from_score_features",
        "passed": True,
        "detail": "is_target is used only for evaluation after ranking.",
    },
    {
        "check_name": "raw_query_review_text_excluded",
        "passed": True,
        "detail": "Raw query text and raw review text are not Shannon score features.",
    },
    {
        "check_name": "recency_features_use_strict_prequery_rule",
        "passed": True,
        "detail": TEMPORAL_LEAKAGE_RULE,
    },
    {
        "check_name": "future_or_equal_prior_rows_excluded",
        "passed": True,
        "detail": json.dumps({"excluded_rows": int(case_user_temporal_maps.get("future_or_equal_excluded_n", 0))}),
    },
    {
        "check_name": "no_prior_model_feature_contract_clean",
        "passed": no_prior_model_feature_contract_clean,
        "detail": NO_PRIOR_MANIFEST_NOTE,
    },
    {
        "check_name": "no_prior_score_components_disabled",
        "passed": no_prior_score_components_disabled,
        "detail": json.dumps({
            "SHANNON_ALPHA": float(SHANNON_ALPHA),
            "SHANNON_TEMPORAL_RECENCY_WEIGHT": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
            "lambda_shannon_disabled": lambda_shannon_disabled,
            "personalization_score_disabled": personalization_score_disabled,
            "shannon_temporal_score_disabled": shannon_temporal_score_disabled,
            "final_score_policy": "shannon_no_prior_rerank preserves original Stage 1 rank",
        }),
    },
    {
        "check_name": "user_prior_temporal_features_excluded",
        "passed": not any(str(c).startswith("user_") for c in temporal_feature_cols),
        "detail": json.dumps([c for c in temporal_feature_cols if str(c).startswith("user_")]),
    },
])

feature_manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "stage": STAGE,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "reranking_method": RERANKING_METHOD,
    "rerank_method": SHANNON_OUTPUT_RERANK_METHOD,
    "experiment_condition": EXPERIMENT_CONDITION,
    "candidate_source": CANDIDATE_SOURCE_LABEL,
    "candidate_pool_type": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "baseline_type": "deterministic_no_prior_shannon_control",
    "feature_registry_version": SHANNON_FEATURE_REGISTRY_VERSION,
    "temporal_feature_version": TEMPORAL_FEATURE_VERSION,
    "temporal_leakage_rule": TEMPORAL_LEAKAGE_RULE,
    "temporal_recency_weight": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
    "temporal_features": temporal_feature_cols,
    "temporal_artifacts_available": bool(temporal_artifacts_available),
    "missing_temporal_paths": missing_temporal_paths,
    "temporal_artifact_paths": {name: str(path) for name, path in TEMPORAL_ARTIFACT_PATHS.items()},
    "temporal_artifact_manifest": temporal_artifact_manifest,
    "policy": "Base no-prior control: Shannon entropy/profile-concentration and user-history temporal score components are disabled; item temporal columns are exported only as diagnostics.",
    "no_prior_manifest_note": NO_PRIOR_MANIFEST_NOTE,
    "no_prior_model_feature_contract_clean": bool(no_prior_model_feature_contract_clean),
    "no_prior_score_components_disabled": bool(no_prior_score_components_disabled),
    "no_prior_score_feature_columns": [],
    "diagnostic_feature_columns": diagnostic_feature_cols,
    "final_score_policy": "No-op control: shannon_no_prior_rerank preserves original Stage 1 baseline rank within each query and pool depth.",
    "ablation_condition": "S2-Q",
    "no_prior_feature_policy": NO_PRIOR_FEATURE_POLICY,
    "no_prior_score_policy": NO_PRIOR_SCORE_POLICY,
    "raw_review_text_used": False,
    "raw_timestamp_columns_used_as_score_features": False,
    "target_timestamp_ms_used_as_metadata_only": True,
}

print("Temporal artifacts available:", bool(temporal_artifacts_available))
print("Temporal feature columns added:", len(temporal_feature_cols))
print("Shannon temporal recency mean:", round(float(feature_base_df["shannon_temporal_recency_score"].mean()), 6))
print("User-prior temporal features used for score:", False)
display(temporal_feature_summary_df.head(30))

# =========================================================
# Validate the no-prior Shannon score and feature contract.
# =========================================================
feature_base_df["shannon_temporal_recency_score"] = safe_numeric(
    feature_base_df["shannon_temporal_recency_score"], default=0.0
).clip(0.0, 1.0).astype("float32")

SHANNON_SCORING_COMPONENTS = (
    ["retrieval_score_norm_pool"]
    if not USE_USER_PRIOR_FEATURES
    else ["retrieval_score_norm_pool", "lambda_shannon", "personalization_score", "shannon_temporal_recency_score"]
)
if USE_USER_PRIOR_FEATURES and "shannon_temporal_recency_score" not in SHANNON_FEATURE_COLUMNS:
    SHANNON_FEATURE_COLUMNS = [*SHANNON_FEATURE_COLUMNS, "shannon_temporal_recency_score"]

SHANNON_FEATURE_DTYPES = {column: str(feature_base_df[column].dtype) for column in SHANNON_FEATURE_COLUMNS}
SHARED_ALL_PRIOR_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank" / "shannon"
SHARED_ALL_PRIOR_CONTRACT_PATH = SHARED_ALL_PRIOR_DIR / "shared_all_prior_shannon_contract.json"
shared_all_prior_contract = {
    "contract_version": "shared_all_prior_shannon_v2_compact_components",
    "category_id": CATEGORY_ID,
    "feature_columns": list(SHANNON_FEATURE_COLUMNS),
    "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
    "scoring_components": list(SHANNON_SCORING_COMPONENTS),
    "profile_roles": list(PROFILE_ROLES),
    "family_strength_weights": dict(FAMILY_STRENGTH_WEIGHTS),
    "shannon_alpha": float(SHANNON_ALPHA),
    "history_saturation_n": int(SHANNON_HISTORY_SATURATION_N),
    "regime_strength_cap": dict(REGIME_STRENGTH_CAP),
    "temporal_recency_weight": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
    "score_formula": "(1-lambda)*retrieval_norm + lambda*((1-recency_weight)*(functional_component+brand_component) + recency_weight*temporal_recency)",
    "tie_break": ["final_score desc", "retrieval_score_norm_pool desc", "candidate_rank asc", "candidate_item_id asc"],
    "missing_value_policy": "zero for absent profile match/count/share/recency; raw brand empty string",
    "cold_user_policy": "zero history-derived features and lambda_shannon=0; no fabricated preference",
    "history_source": "all_prior",
}
feature_columns_S2P = list(SHANNON_FEATURE_COLUMNS)
feature_columns_Full = feature_columns_S2P
feature_dtypes_S2P = dict(SHANNON_FEATURE_DTYPES)
feature_dtypes_Full = feature_dtypes_S2P
feature_schema_equality = feature_columns_S2P == feature_columns_Full and feature_dtypes_S2P == feature_dtypes_Full

if USE_USER_PRIOR_FEATURES and CANDIDATE_POOL_ROLE == "baseline_query_only":
    Path(SHARED_ALL_PRIOR_CONTRACT_PATH).write_text(
        json.dumps(make_jsonable(shared_all_prior_contract), ensure_ascii=False, indent=2), encoding="utf-8"
    )
elif USE_USER_PRIOR_FEATURES and CANDIDATE_POOL_ROLE == "personalized_retrieval":
    expected_shared_contract = load_json_required(SHARED_ALL_PRIOR_CONTRACT_PATH)
    comparable_keys = [
        "contract_version", "category_id", "feature_columns", "feature_dtypes", "scoring_components",
        "profile_roles", "family_strength_weights", "shannon_alpha", "history_saturation_n",
        "regime_strength_cap", "temporal_recency_weight", "score_formula", "tie_break",
        "missing_value_policy", "cold_user_policy", "history_source",
    ]
    mismatched_keys = [
        key for key in comparable_keys if expected_shared_contract.get(key) != shared_all_prior_contract.get(key)
    ]
    if mismatched_keys:
        raise RuntimeError(f"S2-P / Full Shannon contract mismatch: {mismatched_keys}")
    feature_columns_S2P = expected_shared_contract["feature_columns"]
    feature_columns_Full = SHANNON_FEATURE_COLUMNS
    feature_dtypes_S2P = expected_shared_contract["feature_dtypes"]
    feature_dtypes_Full = SHANNON_FEATURE_DTYPES
    feature_schema_equality = feature_columns_S2P == feature_columns_Full and feature_dtypes_S2P == feature_dtypes_Full
    if not feature_schema_equality:
        raise RuntimeError("feature_columns_S2P/Full or feature_dtypes_S2P/Full differ.")

raw_review_columns_in_model_input = sorted({
    column for column in SHANNON_FEATURE_COLUMNS
    if column in {"review_text", "review_body", "raw_review_text", "target_review_text", "heldout_review_text"}
})
if raw_review_columns_in_model_input:
    raise RuntimeError("Raw review columns entered the Shannon feature contract.")
if brand_terms_added_to_synthetic_query_n != 0 or same_target_item_prior_rows != 0 or not temporal_validation_passed:
    raise RuntimeError("A required leakage or temporal contract failed.")

used_model_features_df = pd.DataFrame([
    {
        "feature": feature,
        "feature_group": "direct_score_component" if feature in SHANNON_SCORING_COMPONENTS else "profile_component_or_audit",
        "is_common_model_feature": bool(USE_USER_PRIOR_FEATURES),
        "is_temporal_feature": feature == "shannon_temporal_recency_score",
        "is_dynamic_pool_feature": feature == "retrieval_score_norm_pool",
        "used_in_final_score": feature in SHANNON_SCORING_COMPONENTS or feature.startswith("match__") or feature.startswith("weight__") or feature == "candidate_brand_profile_weight",
    }
    for feature in list(dict.fromkeys([*SHANNON_SCORING_COMPONENTS, *SHANNON_FEATURE_COLUMNS]))
])

brand_contract = {
    "brand_query_enabled": False,
    "brand_candidate_visible": True,
    "brand_reranking_enabled": bool(USE_USER_PRIOR_FEATURES),
    "user_brand_affinity_enabled": bool(USE_USER_PRIOR_FEATURES),
    "history_source": "all_prior" if USE_USER_PRIOR_FEATURES else "none",
    "historical_population_review_signals_in_user_profile": False,
    "candidate_brand_source_column": "brand_facet_text",
    "candidate_brand_non_null_rate": float(candidate_brand_nonnull_rate),
    "notebook04_artifacts_read_directly": False,
}
contract_diagnostics = {
    "candidate_rows": int(len(feature_base_df)),
    "cases": int(feature_base_df["query_id"].nunique()),
    "candidate_brand_non_null_rate": float(candidate_brand_nonnull_rate),
    "users_with_at_least_one_prior_brand": int(users_with_prior_brand),
    "candidate_brand_prior_match_rate": float(candidate_brand_prior_match_rate),
    "cold_user_count": int(query_meta_df["regime"].astype(str).eq("cold").sum()),
    "s2q_feature_count": len(S2Q_FEATURE_COLUMNS),
    "s2p_feature_count": len(SHANNON_FEATURE_COLUMNS) if USE_USER_PRIOR_FEATURES else None,
    "full_feature_count": len(SHANNON_FEATURE_COLUMNS) if USE_USER_PRIOR_FEATURES else None,
    "brand_aware_feature_count": len(USER_BRAND_AFFINITY_COLUMNS) if USE_USER_PRIOR_FEATURES else 0,
    "s2p_full_feature_schema_equality": bool(feature_schema_equality) if USE_USER_PRIOR_FEATURES else None,
    "temporal_validation_status": "pass" if temporal_validation_passed else "fail",
}
feature_manifest.update({
    **brand_contract,
    "scoring_components": list(SHANNON_SCORING_COMPONENTS),
    "feature_columns": list(SHANNON_FEATURE_COLUMNS),
    "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
    "user_brand_feature_columns_s2q": list(user_brand_feature_columns_S2Q),
    "shared_all_prior_contract_path": str(SHARED_ALL_PRIOR_CONTRACT_PATH) if USE_USER_PRIOR_FEATURES else None,
    "contract_diagnostics": contract_diagnostics,
})

print("Shannon scoring components:", SHANNON_SCORING_COMPONENTS)
print("Feature contract columns:", len(SHANNON_FEATURE_COLUMNS))
print("RankP / Full feature-schema equality:", feature_schema_equality if USE_USER_PRIOR_FEATURES else "not applicable")
print("Temporal validation status:", contract_diagnostics["temporal_validation_status"])


## 7. Rerank fixed candidate pool and evaluate

In [ ]:
def score_rank_one_depth(feature_df, pool_depth, rerank_method):
    work = feature_df[feature_df['rank'].le(int(pool_depth))].copy()

    t_score = time.perf_counter()
    work['retrieval_score_norm_pool'] = minmax_by_group(work, 'query_id', 'score', max_rank=int(pool_depth))
    if rerank_method == BASELINE_RERANK_METHOD:
        work['final_score'] = work['retrieval_score_norm_pool']
        work['lambda_applied'] = 0.0
        work['personalization_score_scaled'] = 0.0
        work['temporal_recency_score_scaled'] = 0.0
        work['personalization_score_with_recency'] = 0.0
    elif rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
        work['lambda_applied'] = 0.0
        work['personalization_score_scaled'] = 0.0
        work['temporal_recency_score_scaled'] = 0.0
        work['personalization_score_with_recency'] = 0.0
        work['final_score'] = work['retrieval_score_norm_pool']
    else:
        raise ValueError(f'Unknown rerank_method: {rerank_method}')
    scoring_runtime_sec = time.perf_counter() - t_score

    t_sort = time.perf_counter()
    work = work.sort_values(['query_id', 'rank', 'item_id'], ascending=[True, True, True]).copy()
    work['new_rank'] = work.groupby('query_id').cumcount() + 1
    work['rank_shift'] = work['rank'].astype(int) - work['new_rank'].astype(int)
    work['pool_depth'] = int(pool_depth)
    work['rerank_method'] = rerank_method
    sorting_runtime_sec = time.perf_counter() - t_sort
    return work, scoring_runtime_sec, sorting_runtime_sec


def per_query_metrics_for_ranked_frame(ranked_df, rank_col, rerank_method, pool_depth):
    base_cols = [
        'case_id', 'query_id', 'user_id', 'regime', 'sampling_bracket', 'target_selection_mode',
        'query_method', 'query_text', 'target_item_id', 'gt_item_id', 'prior_review_n', 'prior_item_n',
        'user_total_reviews', 'query_token_len', 'removed_token_count', 'candidate_pool_type', 'retrieval_method', 'retrieval_method_label',
    ]
    base_cols = [c for c in base_cols if c in query_meta_df.columns]
    base = query_meta_df[base_cols].drop_duplicates('query_id').copy()

    target_ranks = (
        ranked_df[ranked_df['is_target'].astype(bool)]
        .groupby('query_id')[rank_col]
        .min()
        .rename('gt_rank')
        .reset_index()
    )
    out = base.merge(target_ranks, on='query_id', how='left')
    out['pool_depth'] = int(pool_depth)
    out['rerank_method'] = rerank_method
    out['gt_in_pool'] = out['gt_rank'].notna().astype(int)

    metric_rows = [metrics_at_rank(rank, EVAL_KS) for rank in out['gt_rank'].tolist()]
    metrics_df = pd.DataFrame(metric_rows)
    out = pd.concat([out.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

    profile_keep = [
        'query_id', 'profile_history_review_n', 'profile_history_item_n', 'user_entropy_norm_shannon',
        'profile_concentration', 'history_factor', 'lambda_shannon',
    ]
    out = out.merge(query_profiles_df[profile_keep], on='query_id', how='left', validate='one_to_one')
    out['lambda_applied'] = 0.0

    # Preference-alignment diagnostics at the user-facing top-5.
    top5 = ranked_df[ranked_df['new_rank'].le(PREFERENCE_ALIGNMENT_K)].sort_values(['query_id', 'new_rank'])
    top_items_by_query = top5.groupby('query_id')['item_id'].apply(list).to_dict()
    supplement_families = [f for f in SUPPLEMENTARY_FAMILY_PRIORITY if f in FACIAL_ATTRIBUTE_FAMILIES]
    supplement_weights = {f: global_family_weights.get(f, 0.0) for f in supplement_families}
    supplementary_rows = [
        weighted_similarity_to_target(
            top_items_by_query.get(str(row.query_id), []),
            str(row.gt_item_id),
            supplement_families,
            family_weights=supplement_weights,
            k=PREFERENCE_ALIGNMENT_K,
        )
        for row in out.itertuples(index=False)
    ]
    out = pd.concat([out.reset_index(drop=True), pd.DataFrame(supplementary_rows).reset_index(drop=True)], axis=1)
    return out


def summarize_metrics(df, group_cols):
    metric_cols = [c for c in df.columns if re.match(r'^(HitRate|MRR|NDCG)@\d+$', str(c))]
    agg_map = {'query_id': 'nunique'}
    for col in metric_cols + [c for c in SUPPLEMENTARY_METRIC_COLS if c in df.columns]:
        agg_map[col] = 'mean'
    out = df.groupby(group_cols, dropna=False).agg(agg_map).reset_index().rename(columns={'query_id': 'n_queries'})
    if 'regime' in out.columns:
        out = apply_regime_order(out, 'regime')
    sort_cols = [c for c in group_cols if c in out.columns]
    return out.sort_values(sort_cols).reset_index(drop=True)

print('Reranking and evaluation helpers ready.')

In [ ]:
candidate_frames = []
per_query_frames = []
runtime_pool_depth_rows = []

for pool_depth in POOL_DEPTHS:
    depth_n_candidates = int(feature_base_df['rank'].le(int(pool_depth)).sum())

    for rerank_method in EXPECTED_RERANK_METHODS:
        ranked_df, scoring_runtime_sec, sorting_runtime_sec = score_rank_one_depth(feature_base_df, pool_depth, rerank_method)

        eval_start = time.perf_counter()
        per_query_df = per_query_metrics_for_ranked_frame(ranked_df, 'new_rank', rerank_method, pool_depth)
        eval_runtime_sec = time.perf_counter() - eval_start

        candidate_frames.append(ranked_df)
        per_query_frames.append(per_query_df)

        if rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
            method_runtime_for_online = scoring_runtime_sec + sorting_runtime_sec
            runtime_pool_depth_rows.append({
                'category_id': CATEGORY_ID,
                'category_folder': CATEGORY_FOLDER,
                'method': RERANKING_METHOD,
                'reranker': rerank_method,
                'pool_depth': int(pool_depth),
                'candidate_pool_depth': int(pool_depth),
                'n_queries': int(ranked_df['query_id'].nunique()),
                'n_candidates': int(len(ranked_df)),
                'feature_preparation_runtime_sec': np.nan,
                'model_scoring_runtime_sec': float(scoring_runtime_sec),
                'ranking_sorting_runtime_sec': float(sorting_runtime_sec),
                'online_operation_runtime_sec': float(method_runtime_for_online),
                'runtime_sec_per_query': float(method_runtime_for_online / max(ranked_df['query_id'].nunique(), 1)),
                'runtime_sec_per_candidate': float(method_runtime_for_online / max(len(ranked_df), 1)),
                'queries_per_second': float(ranked_df['query_id'].nunique() / method_runtime_for_online) if method_runtime_for_online > 0 else np.nan,
                'candidates_per_second': float(len(ranked_df) / method_runtime_for_online) if method_runtime_for_online > 0 else np.nan,
                'rerank_runtime_sec': float(method_runtime_for_online),
                'rerank_runtime_sec_per_query': float(method_runtime_for_online / max(ranked_df['query_id'].nunique(), 1)),
                'rerank_runtime_sec_per_candidate': float(method_runtime_for_online / max(len(ranked_df), 1)),
                'eval_runtime_sec': float(eval_runtime_sec),
                'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
                'candidate_pool_type': candidate_pool_type_actual,
                'retrieval_method': retrieval_method_actual,
                'retrieval_method_label': retrieval_method_label_actual,
                'reranking_method': RERANKING_METHOD,
            })

        # Record the common wall-clock step rows at the report depth.
        if int(pool_depth) == REPORT_POOL_DEPTH and rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
            record_runtime_step('model_scoring', scoring_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))
            record_runtime_step('ranking_sorting', sorting_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))
            record_runtime_step('evaluation', eval_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))

reranked_candidates_all_depths_df = pd.concat(candidate_frames, ignore_index=True)
per_query_results_df = pd.concat(per_query_frames, ignore_index=True)
runtime_by_pool_depth_df = pd.DataFrame(runtime_pool_depth_rows)

baseline_order_df = (
    reranked_candidates_all_depths_df[reranked_candidates_all_depths_df["rerank_method"].astype(str).eq(BASELINE_RERANK_METHOD)]
    [["query_id", "pool_depth", "item_id", "new_rank"]]
    .rename(columns={"new_rank": "baseline_new_rank"})
)
shannon_order_df = (
    reranked_candidates_all_depths_df[reranked_candidates_all_depths_df["rerank_method"].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]
    [["query_id", "pool_depth", "item_id", "new_rank"]]
    .rename(columns={"new_rank": "shannon_new_rank"})
)
ranking_equivalence_df = shannon_order_df.merge(
    baseline_order_df,
    on=["query_id", "pool_depth", "item_id"],
    how="outer",
    validate="one_to_one",
)
ranking_equivalence_mismatch_n = int(
    ranking_equivalence_df["baseline_new_rank"].isna().sum()
    + ranking_equivalence_df["shannon_new_rank"].isna().sum()
    + ranking_equivalence_df["baseline_new_rank"].ne(ranking_equivalence_df["shannon_new_rank"]).fillna(True).sum()
)
no_prior_shannon_ranking_matches_stage1_baseline = ranking_equivalence_mismatch_n == 0
if not no_prior_shannon_ranking_matches_stage1_baseline:
    examples = ranking_equivalence_df[
        ranking_equivalence_df["baseline_new_rank"].ne(ranking_equivalence_df["shannon_new_rank"]).fillna(True)
    ].head(10).to_dict("records")
    raise RuntimeError(f"No-prior Shannon ranking differs from Stage 1 baseline: {examples}")


# Ensure the per-depth runtime export has the common online-runtime schema.
if not runtime_by_pool_depth_df.empty:
    if "candidate_pool_depth" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["candidate_pool_depth"] = pd.to_numeric(runtime_by_pool_depth_df["pool_depth"], errors="coerce").astype("Int64")
    if "method" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["method"] = RERANKING_METHOD
    if "online_operation_runtime_sec" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["online_operation_runtime_sec"] = runtime_by_pool_depth_df.get("rerank_runtime_sec", np.nan)
    if "runtime_sec_per_query" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_query"] = runtime_by_pool_depth_df.get("rerank_runtime_sec_per_query", np.nan)
    if "runtime_sec_per_candidate" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_candidate"] = runtime_by_pool_depth_df.get("rerank_runtime_sec_per_candidate", np.nan)
    if "queries_per_second" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["queries_per_second"] = np.where(
            pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce") > 0,
            pd.to_numeric(runtime_by_pool_depth_df["n_queries"], errors="coerce") / pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce"),
            np.nan,
        )
    if "candidates_per_second" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["candidates_per_second"] = np.where(
            pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce") > 0,
            pd.to_numeric(runtime_by_pool_depth_df["n_candidates"], errors="coerce") / pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce"),
            np.nan,
        )
    if "feature_preparation_runtime_sec" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["feature_preparation_runtime_sec"] = np.nan
    if "model_fit_or_tuning_runtime_sec" not in runtime_by_pool_depth_df.columns and "fitting_runtime_sec" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["model_fit_or_tuning_runtime_sec"] = runtime_by_pool_depth_df["fitting_runtime_sec"]


# Report-depth candidate export only, with both baseline and Shannon rows.
reranked_candidates_report_df = reranked_candidates_all_depths_df[reranked_candidates_all_depths_df['pool_depth'].eq(REPORT_POOL_DEPTH)].copy()
per_query_report_df = per_query_results_df[per_query_results_df['pool_depth'].eq(REPORT_POOL_DEPTH)].copy()

executed_methods = sorted(per_query_results_df['rerank_method'].astype(str).unique().tolist())
expected_pool_depths = sorted([int(x) for x in POOL_DEPTHS])
observed_pool_depths = sorted(pd.to_numeric(per_query_results_df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist())
if executed_methods != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError(f'Unexpected rerank methods: {executed_methods}')
if observed_pool_depths != expected_pool_depths:
    raise RuntimeError(f'Pool-depth outputs are incomplete. Expected {expected_pool_depths}, found {observed_pool_depths}')

# Candidate-set preservation check for full depth.
before_set = candidate_df.groupby('query_id')['item_id'].apply(lambda s: tuple(sorted(s.astype(str))))
after_set = (
    reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].eq(SHANNON_OUTPUT_RERANK_METHOD)]
    .groupby('query_id')['item_id']
    .apply(lambda s: tuple(sorted(s.astype(str))))
)
candidate_set_changed_by_reranker = not before_set.equals(after_set)
if candidate_set_changed_by_reranker:
    raise RuntimeError('Shannon reranker changed the report-depth candidate set.')

print('per_query_results_df shape:', per_query_results_df.shape)
print('reranked_candidates_report_df shape:', reranked_candidates_report_df.shape)
print('runtime_by_pool_depth_df shape:', runtime_by_pool_depth_df.shape)
print('Executed methods:', executed_methods)
display(runtime_by_pool_depth_df.head(20))

## 8. Aggregate summaries, uplift, and diagnostics

In [ ]:
def add_summary_scope_metadata(df, summary_scope):
    work = df.copy()
    work['report_pool_depth'] = REPORT_POOL_DEPTH
    work['summary_scope'] = summary_scope
    work['condition'] = EXPERIMENT_CONDITION
    work['candidate_source'] = CANDIDATE_SOURCE_LABEL
    work['candidate_source_format'] = CANDIDATE_SOURCE_FORMAT
    work['candidate_pool_type'] = candidate_pool_type_actual
    work['retrieval_method'] = retrieval_method_actual
    work['retrieval_method_label'] = retrieval_method_label_actual
    work['reranking_method'] = RERANKING_METHOD
    work['comparison_set'] = 'native'
    work['category_id'] = CATEGORY_ID
    work['category_folder'] = CATEGORY_FOLDER
    work['category_label'] = CATEGORY_LABEL
    work['primary_metric'] = PRIMARY_STAGE2_METRIC
    return work


def build_uplift_summary(df, group_cols):
    group_cols = list(group_cols)
    ranking_metric_cols = [c for c in df.columns if re.match(r'^(HitRate|MRR|NDCG)@\d+$', str(c))]
    diagnostic_metric_cols = [c for c in UPLIFT_DIAGNOSTIC_METRICS if c in df.columns]
    metric_cols = ranking_metric_cols + diagnostic_metric_cols
    key_cols = group_cols + ['query_id']
    baseline = df[df['rerank_method'].astype(str).eq(BASELINE_RERANK_METHOD)][key_cols + metric_cols].copy()
    shannon = df[df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)][key_cols + metric_cols].copy()
    merged = shannon.merge(baseline, on=key_cols, how='inner', suffixes=('_shannon', '_baseline'), validate='one_to_one')
    for col in metric_cols:
        merged[f'delta_{col}'] = pd.to_numeric(merged[f'{col}_shannon'], errors='coerce').fillna(0.0) - pd.to_numeric(merged[f'{col}_baseline'], errors='coerce').fillna(0.0)
    delta_cols = [c for c in merged.columns if c.startswith('delta_')]
    if group_cols:
        out = merged.groupby(group_cols, dropna=False).agg({'query_id': 'nunique', **{c: 'mean' for c in delta_cols}}).reset_index().rename(columns={'query_id': 'n_queries'})
    else:
        out = pd.DataFrame([{**{'n_queries': int(merged['query_id'].nunique())}, **{c: float(merged[c].mean()) for c in delta_cols}}])
    return merged, out


summary_overall_df = add_summary_scope_metadata(summarize_metrics(per_query_report_df, ['rerank_method']), f'pool_depth_{REPORT_POOL_DEPTH}')
summary_by_pool_depth_df = add_summary_scope_metadata(summarize_metrics(per_query_results_df, ['pool_depth', 'rerank_method']), 'all_pool_depths')
summary_by_regime_df = add_summary_scope_metadata(summarize_metrics(per_query_report_df, ['regime', 'rerank_method']), f'pool_depth_{REPORT_POOL_DEPTH}')
summary_by_regime_pool_depth_df = add_summary_scope_metadata(summarize_metrics(per_query_results_df, ['regime', 'pool_depth', 'rerank_method']), 'all_pool_depths')

# Attach runtime to by-depth summary for quick efficiency inspection.
runtime_merge_cols = ['pool_depth', 'reranker', 'rerank_runtime_sec', 'rerank_runtime_sec_per_query', 'rerank_runtime_sec_per_candidate', 'eval_runtime_sec']
summary_by_pool_depth_df = summary_by_pool_depth_df.merge(
    runtime_by_pool_depth_df[runtime_merge_cols],
    left_on=['pool_depth', 'rerank_method'],
    right_on=['pool_depth', 'reranker'],
    how='left',
).drop(columns=['reranker'], errors='ignore')

overall_delta_detail_df, uplift_overall_df = build_uplift_summary(per_query_report_df, [])
pool_delta_detail_df, uplift_by_pool_depth_df = build_uplift_summary(per_query_results_df, ['pool_depth'])
regime_delta_detail_df, uplift_by_regime_df = build_uplift_summary(per_query_report_df, ['regime'])
regime_pool_delta_detail_df, uplift_by_regime_pool_depth_df = build_uplift_summary(per_query_results_df, ['regime', 'pool_depth'])

# Report-scope QC for downstream aggregation.
def pool_depth_value_string(df):
    if df is None or df.empty or 'pool_depth' not in df.columns:
        return ''
    vals = sorted(pd.to_numeric(df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist())
    return ','.join(map(str, vals))


def qc_row(output_file, expected_scope, source_dataframe, df):
    return {
        'notebook_name': NOTEBOOK_NAME,
        'output_file': output_file,
        'expected_scope': expected_scope,
        'actual_scope': expected_scope,
        'source_dataframe': source_dataframe,
        'pool_depth_values': pool_depth_value_string(df),
        'n_rows': int(len(df)) if df is not None else 0,
        'n_queries': int(df['query_id'].nunique()) if df is not None and 'query_id' in df.columns else int(pd.to_numeric(df['n_queries'], errors='coerce').max()) if df is not None and 'n_queries' in df.columns and len(df) else 0,
        'check_passed': True,
        'warning_message': '',
    }

report_summary_scope_qc_df = pd.DataFrame([
    qc_row('results_overall.csv', f'pool_depth_{REPORT_POOL_DEPTH}', 'per_query_report_df', per_query_report_df),
    qc_row('results_by_regime.csv', f'pool_depth_{REPORT_POOL_DEPTH}', 'per_query_report_df', per_query_report_df),
    qc_row('results_by_pool_depth.csv', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
    qc_row('results_by_regime_pool_depth.csv', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
    qc_row('per_query_metrics.parquet', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
])

contract_qc_rows = pd.DataFrame([
    {
        "notebook_name": NOTEBOOK_NAME,
        "output_file": "feature_leakage_qc.csv",
        "expected_scope": "score_contract",
        "actual_scope": "score_contract",
        "source_dataframe": "feature_leakage_qc_df",
        "pool_depth_values": "",
        "n_rows": int(len(feature_leakage_qc_df)) if "feature_leakage_qc_df" in globals() else 0,
        "n_queries": int(n_queries),
        "check_passed": bool(no_prior_model_feature_contract_clean) if "no_prior_model_feature_contract_clean" in globals() else False,
        "warning_message": "" if globals().get("no_prior_model_feature_contract_clean", False) else "No-prior score contract failed.",
        "check_name": "no_prior_model_feature_contract_clean",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "output_file": "feature_leakage_qc.csv",
        "expected_scope": "score_contract",
        "actual_scope": "score_contract",
        "source_dataframe": "feature_leakage_qc_df",
        "pool_depth_values": "",
        "n_rows": int(len(feature_leakage_qc_df)) if "feature_leakage_qc_df" in globals() else 0,
        "n_queries": int(n_queries),
        "check_passed": bool(no_prior_score_components_disabled) if "no_prior_score_components_disabled" in globals() else False,
        "warning_message": "" if globals().get("no_prior_score_components_disabled", False) else "No-prior score components are not fully disabled.",
        "check_name": "no_prior_score_components_disabled",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "output_file": "reranked_candidates.parquet",
        "expected_scope": "all_pool_depths",
        "actual_scope": "all_pool_depths",
        "source_dataframe": "ranking_equivalence_df",
        "pool_depth_values": pool_depth_value_string(ranking_equivalence_df) if "pool_depth" in ranking_equivalence_df.columns else "",
        "n_rows": int(len(ranking_equivalence_df)) if "ranking_equivalence_df" in globals() else 0,
        "n_queries": int(ranking_equivalence_df["query_id"].nunique()) if "ranking_equivalence_df" in globals() else 0,
        "check_passed": bool(no_prior_shannon_ranking_matches_stage1_baseline) if "no_prior_shannon_ranking_matches_stage1_baseline" in globals() else False,
        "warning_message": "" if globals().get("no_prior_shannon_ranking_matches_stage1_baseline", False) else "No-prior Shannon ranking differs from Stage 1 baseline.",
        "check_name": "no_prior_shannon_ranking_matches_stage1_baseline",
    },
])
report_summary_scope_qc_df = pd.concat([report_summary_scope_qc_df, contract_qc_rows], ignore_index=True)


feature_diagnostics_df = pd.DataFrame([
    {
        'feature': col,
        'mean': float(pd.to_numeric(feature_base_df[col], errors='coerce').mean()),
        'std': float(pd.to_numeric(feature_base_df[col], errors='coerce').std()),
        'min': float(pd.to_numeric(feature_base_df[col], errors='coerce').min()),
        'max': float(pd.to_numeric(feature_base_df[col], errors='coerce').max()),
    }
    for col in ['retrieval_score_norm_full_pool', 'personalization_score', 'lambda_shannon'] + [f'match__{f}' for f in FACIAL_ATTRIBUTE_FAMILIES]
    if col in feature_base_df.columns
])

feature_importance_df = global_family_weights_df.rename(columns={'mean_profile_weight': 'importance'}).copy()
feature_importance_df['model_type'] = 'deterministic_shannon'
feature_importance_df['importance_source'] = 'mean_profile_family_weight'
feature_importance_summary_df = feature_importance_df[['family', 'importance', 'mean_match', 'global_family_weight', 'item_support', 'vocab_size', 'model_type', 'importance_source']].copy()

tuning_trials_df = pd.DataFrame([{
    'trial_id': 'deterministic_reference',
    'trial_source': 'fixed_heuristic',
    'selected': True,
    'offline_tuning_used': False,
    'shannon_alpha': SHANNON_ALPHA,
    'history_saturation_n': SHANNON_HISTORY_SATURATION_N,
    'cold_cap': REGIME_STRENGTH_CAP['cold'],
    'weak_cap': REGIME_STRENGTH_CAP['weak'],
    'moderate_cap': REGIME_STRENGTH_CAP['moderate'],
    'strong_cap': REGIME_STRENGTH_CAP['strong'],
}])
tuning_summary = {
    'offline_tuning_used': False,
    'method_type': 'deterministic_shannon_entropy_heuristic',
    'selected_trial': make_jsonable(tuning_trials_df.iloc[0].to_dict()),
}

query_history_summary_df = pd.DataFrame([
    {
        'summary_name': 'query_profiles',
        'n_queries': int(query_profiles_df['query_id'].nunique()),
        'n_rows': int(len(query_profiles_df)),
        'profile_history_review_n_mean': float(query_profiles_df['profile_history_review_n'].mean()),
        'profile_history_item_n_mean': float(query_profiles_df['profile_history_item_n'].mean()),
    },
    {
        'summary_name': 'prior_history',
        'n_queries': int(prior_history_df['case_id'].nunique()),
        'n_rows': int(len(prior_history_df)),
        'profile_history_review_n_mean': float(prior_history_df.groupby('case_id').size().mean()) if len(prior_history_df) else 0.0,
        'profile_history_item_n_mean': float(prior_history_df.groupby('case_id')['prior_item_id'].nunique().mean()) if len(prior_history_df) else 0.0,
    },
])

print('Overall summary:')
display(summary_overall_df)
print('Uplift summary:')
display(uplift_overall_df)
print('By-depth summary preview:')
display(summary_by_pool_depth_df.head(20))

## 9. Export canonical outputs

In [ ]:
# Canonical output paths expected by downstream Stage 2 comparison notebooks.

saved_paths = []

results_overall_path = OUT_DIR / 'results_overall.csv'
results_by_pool_depth_path = OUT_DIR / 'results_by_pool_depth.csv'
results_by_regime_path = OUT_DIR / 'results_by_regime.csv'
results_by_regime_pool_depth_path = OUT_DIR / 'results_by_regime_pool_depth.csv'
runtime_by_pool_depth_path = OUT_DIR / 'runtime_by_pool_depth.csv'
uplift_summary_path = OUT_DIR / 'uplift_summary.csv'
uplift_by_pool_depth_path = OUT_DIR / 'uplift_by_pool_depth.csv'
uplift_by_regime_path = OUT_DIR / 'uplift_by_regime.csv'
uplift_by_regime_pool_depth_path = OUT_DIR / 'uplift_by_regime_pool_depth.csv'
feature_table_path = OUT_DIR / 'feature_table.parquet'
per_query_results_path = OUT_DIR / 'per_query_metrics.parquet'
reranked_candidates_path = OUT_DIR / 'reranked_candidates.parquet'
query_profiles_path = OUT_DIR / 'user_profiles.parquet'
tuning_trials_path = OUT_DIR / 'tuning_trials.csv'
tuning_summary_json_path = OUT_DIR / 'tuning_summary.json'
query_history_summary_path = OUT_DIR / 'query_history_summary.csv'
global_family_weights_path = OUT_DIR / 'global_family_weights.csv'
feature_diagnostics_path = OUT_DIR / 'feature_diagnostics.csv'
feature_importance_path = OUT_DIR / 'feature_importance.csv'
feature_importance_summary_path = OUT_DIR / 'feature_importance_summary.csv'
config_snapshot_path = OUT_DIR / 'config_snapshot.json'
diagnostics_summary_path = OUT_DIR / 'diagnostics_summary.csv'
report_summary_scope_qc_path = OUT_DIR / 'report_summary_scope_qc.csv'
run_manifest_path = OUT_DIR / 'run_manifest.json'

export_material_start = time.perf_counter()

summary_overall_df.to_csv(results_overall_path, index=False)
summary_by_pool_depth_df.to_csv(results_by_pool_depth_path, index=False)
summary_by_regime_df.to_csv(results_by_regime_path, index=False)
summary_by_regime_pool_depth_df.to_csv(results_by_regime_pool_depth_path, index=False)
runtime_by_pool_depth_df.to_csv(runtime_by_pool_depth_path, index=False)
uplift_overall_df.to_csv(uplift_summary_path, index=False)
uplift_by_pool_depth_df.to_csv(uplift_by_pool_depth_path, index=False)
uplift_by_regime_df.to_csv(uplift_by_regime_path, index=False)
uplift_by_regime_pool_depth_df.to_csv(uplift_by_regime_pool_depth_path, index=False)

feature_export_cols = [
    "case_id", "query_id", "user_id", "regime", "item_id", "target_item_id", "gt_item_id",
    "rank", "score", "candidate_brand_raw", "retrieval_score_norm_full_pool",
    "profile_history_review_n", "profile_history_item_n", "user_entropy_norm_shannon",
    "profile_concentration", "shannon_feature_available",
] + list(SHANNON_FEATURE_COLUMNS)
feature_export_cols = list(dict.fromkeys(c for c in feature_export_cols if c in feature_base_df.columns))
feature_base_df[feature_export_cols].to_parquet(feature_table_path, index=False)

per_query_results_df.to_parquet(per_query_results_path, index=False)
reranked_candidates_report_df.to_parquet(reranked_candidates_path, index=False)
query_profiles_df.to_parquet(query_profiles_path, index=False)
tuning_trials_df.to_csv(tuning_trials_path, index=False)
query_history_summary_df.to_csv(query_history_summary_path, index=False)
global_family_weights_df.to_csv(global_family_weights_path, index=False)
feature_diagnostics_df.to_csv(feature_diagnostics_path, index=False)
feature_importance_df.to_csv(feature_importance_path, index=False)
feature_importance_summary_df.to_csv(feature_importance_summary_path, index=False)
report_summary_scope_qc_df.to_csv(report_summary_scope_qc_path, index=False)
with open(tuning_summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(make_jsonable(tuning_summary), f, ensure_ascii=False, indent=2)

config_snapshot = {
    'notebook_name': NOTEBOOK_NAME,
    'stage': STAGE,
    'category_id': CATEGORY_ID,
    'category_folder': CATEGORY_FOLDER,
    'category_label': CATEGORY_LABEL,
    'experiment_condition': EXPERIMENT_CONDITION,
    'condition': EXPERIMENT_CONDITION,
    'no_prior_manifest_note': NO_PRIOR_MANIFEST_NOTE,
    'no_prior_score_policy': NO_PRIOR_SCORE_POLICY,
    'no_prior_model_feature_contract_clean': bool(no_prior_model_feature_contract_clean),
    'no_prior_score_components_disabled': bool(no_prior_score_components_disabled),
    'no_prior_shannon_ranking_matches_stage1_baseline': bool(no_prior_shannon_ranking_matches_stage1_baseline),
    'candidate_source': CANDIDATE_SOURCE_LABEL,
    'candidate_source_format': CANDIDATE_SOURCE_FORMAT,
    'project_root': str(PROJECT_ROOT),
    'output_dir': str(OUT_DIR),
    'candidate_pool_path': str(CANDIDATE_POOL_PATH),
    'candidate_pool_type': candidate_pool_type_actual,
    'retrieval_method': retrieval_method_actual,
    'retrieval_method_label': retrieval_method_label_actual,
    'reranking_method': RERANKING_METHOD,
    'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
    'query_method': STAGE1_QUERY_METHOD,
    'pool_depths': [int(x) for x in POOL_DEPTHS],
    'report_pool_depth': int(REPORT_POOL_DEPTH),
    'eval_ks': [int(x) for x in EVAL_KS],
    'primary_stage2_metric': PRIMARY_STAGE2_METRIC,
    'primary_stage2_metrics': PRIMARY_STAGE2_METRICS,
    'primary_metrics': PRIMARY_STAGE2_METRICS,
    'secondary_stage2_metrics': SECONDARY_STAGE2_METRICS,
    'secondary_metrics': SECONDARY_STAGE2_METRICS,
    'supplementary_stage2_metrics': SUPPLEMENTARY_STAGE2_METRICS,
    'common_comparison_metrics': COMMON_STAGE2_COMPARISON_METRICS,
    'preference_diagnostic_metrics': PREFERENCE_DIAGNOSTIC_METRICS,
    'preference_diagnostic_families': PREFERENCE_DIAGNOSTIC_FAMILIES,
    'preference_diagnostic_family_weights': PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS,
    'preference_alignment_k': int(PREFERENCE_ALIGNMENT_K),
    'expected_rerank_methods': list(EXPECTED_RERANK_METHODS),
    'deterministic_heuristic': True,
    'deterministic_no_prior_control': True,
    'supervised_reranker': False,
    'final_score_policy': 'No-op control preserving original Stage 1 baseline rank.',
    'shannon_alpha': float(SHANNON_ALPHA),
    'shannon_temporal_recency_weight': float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
    'offline_tuning_used': False,

    'item_facet_evidence_policy': ITEM_FACET_EVIDENCE_POLICY,
    'item_facet_source_qc': ITEM_FACET_SOURCE_QC,
    'entropy_normalization_policy': ENTROPY_NORMALIZATION_POLICY,
    'shannon_scoring_policy': SHANNON_SCORING_POLICY,
    'algorithm_change_scope': ALGORITHM_CHANGE_SCOPE,
    'raw_reviews_loaded': False,
    'target_review_text_used_or_saved': False,
    'strict_prior_history_loaded': False,
    'same_item_prior_policy': 'not used for no-prior reranking; prior-history-derived score features are disabled',
    'runtime_framework': 'wall_clock_step_plus_component_summary',
}
Path(config_snapshot_path).write_text(json.dumps(make_jsonable(config_snapshot), ensure_ascii=False, indent=2), encoding='utf-8')

saved_paths.extend([
    results_overall_path, results_by_pool_depth_path, results_by_regime_path, results_by_regime_pool_depth_path,
    runtime_by_pool_depth_path, uplift_summary_path, uplift_by_pool_depth_path, uplift_by_regime_path,
    uplift_by_regime_pool_depth_path, feature_table_path, per_query_results_path, reranked_candidates_path,
    query_profiles_path, tuning_trials_path, tuning_summary_json_path, query_history_summary_path,
    global_family_weights_path, feature_diagnostics_path, feature_importance_path, feature_importance_summary_path,
    report_summary_scope_qc_path, config_snapshot_path,
])

# Runtime export is called last so export_outputs and total_notebook include material-output writing.
export_material_runtime_sec = time.perf_counter() - export_material_start
export_n_candidates_report = int(len(reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]))
record_runtime_step(
    'export_outputs',
    export_material_runtime_sec,
    pool_depth=REPORT_POOL_DEPTH,
    n_queries=n_queries,
    n_candidates=export_n_candidates_report,
)
runtime_steps_df, runtime_notebook_summary_df, runtime_method_summary_at1000_df, runtime_pool_depth_diagnostic_df, runtime_method_components_at1000_df = export_runtime_logs(OUT_DIR, report_pool_depth=REPORT_POOL_DEPTH)
for runtime_path in [
    OUT_DIR / 'runtime_steps.csv',
    OUT_DIR / 'runtime_notebook_summary.csv',
    OUT_DIR / 'runtime_method_summary_at1000.csv',
    OUT_DIR / 'runtime_pool_depth_diagnostic.csv',
    OUT_DIR / 'runtime_method_components_at1000_face.csv',
]:
    saved_paths.append(runtime_path)

required_input_paths = {
    "stage1_candidate_pool_manifest": CANDIDATE_POOL_MANIFEST_PATH,
    "candidate_pool_path": CANDIDATE_POOL_PATH,
    "query_cache_parquet": QUERY_CACHE_PARQUET,
    "items_facets_parquet": ITEMS_FACETS_PARQUET,
}
if USE_USER_PRIOR_FEATURES:
    required_input_paths["prior_history_parquet"] = PRIOR_HISTORY_PARQUET

optional_input_paths = {
    "query_cache_summary_path": QUERY_CACHE_SUMMARY_PATH,
    "query_cache_config_path": QUERY_CACHE_CONFIG_PATH,
    "item_schema_parquet": ITEM_SCHEMA_PARQUET,
    "item_schema_base_parquet": ITEM_SCHEMA_BASE_PARQUET,
    "item_docs_parquet": ITEM_DOCS_PARQUET,
}

input_paths = {
    **{name: str(path) for name, path in required_input_paths.items()},
    **{name: str(path) for name, path in optional_input_paths.items()},
}

all_output_paths = saved_paths + [run_manifest_path, diagnostics_summary_path]

run_manifest = {
    **config_snapshot,
    'input_paths': input_paths,
    'output_paths': {Path(p).stem: str(p) for p in all_output_paths},
    'query_count': int(n_queries),
    'candidate_rows': int(n_candidates),
    'candidate_rows_report_depth': int(len(reranked_candidates_report_df)),
    'executed_rerank_methods': list(executed_methods),
    'candidate_set_changed_by_reranker': bool(candidate_set_changed_by_reranker),
    'regime_counts': query_meta_df['regime'].astype(str).value_counts().to_dict(),
    'created_outputs': [str(p) for p in all_output_paths],
}
Path(run_manifest_path).write_text(json.dumps(make_jsonable(run_manifest), ensure_ascii=False, indent=2), encoding='utf-8')
saved_paths.append(run_manifest_path)

diagnostic_rows = []
for name, path in required_input_paths.items():
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'input_exists__{name}',
        'path': str(path),
        'required': True,
        'check_passed': exists,
        'file_exists': exists,
        'warning_message': '' if exists else 'Required input missing.',
    })
for name, path in optional_input_paths.items():
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'input_exists__{name}',
        'path': str(path),
        'required': False,
        'check_passed': True,
        'file_exists': exists,
        'warning_message': '' if exists else 'Optional input not found; not required for this run.',
    })
for path in saved_paths:
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'output_written__{path.name}',
        'path': str(path),
        'required': True,
        'check_passed': exists,
        'file_exists': exists,
        'warning_message': '' if exists else 'Expected output missing.',
    })
diagnostic_rows.append({
    'stage': STAGE,
    'check_name': 'candidate_set_unchanged_by_reranker',
    'path': '',
    'required': True,
    'check_passed': not candidate_set_changed_by_reranker,
    'file_exists': True,
    'warning_message': '' if not candidate_set_changed_by_reranker else 'Candidate set changed after reranking.',
})
metadata_qc_checks = {
    'retrieval_method_matches_expected': retrieval_method_actual == CANDIDATE_RETRIEVAL_METHOD_LABEL,
    'retrieval_method_label_matches_expected': retrieval_method_label_actual == CANDIDATE_RETRIEVAL_METHOD_LABEL,
    'candidate_pool_type_matches_expected': candidate_pool_type_actual == CANDIDATE_RETRIEVAL_METHOD_LABEL,
    'reranking_method_is_shannon_no_prior': RERANKING_METHOD == "shannon_no_prior",
}
for check_name, check_passed in metadata_qc_checks.items():
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': check_name,
        'path': '',
        'required': True,
        'check_passed': bool(check_passed),
        'file_exists': True,
        'warning_message': '' if check_passed else 'Metadata value did not match expected constant.',
    })
diagnostics_summary_df = pd.DataFrame(diagnostic_rows)
diagnostics_summary_df.to_csv(diagnostics_summary_path, index=False)
diagnostics_exists = diagnostics_summary_path.exists()
diagnostic_rows.append({
    'stage': STAGE,
    'check_name': f'output_written__{diagnostics_summary_path.name}',
    'path': str(diagnostics_summary_path),
    'required': True,
    'check_passed': diagnostics_exists,
    'file_exists': diagnostics_exists,
    'warning_message': '' if diagnostics_exists else 'Expected output missing.',
})
diagnostics_summary_df = pd.DataFrame(diagnostic_rows)
diagnostics_summary_df.to_csv(diagnostics_summary_path, index=False)
saved_paths.append(diagnostics_summary_path)

stale_source_method_key = (
    SOURCE_BASELINE_METHOD_KEY
    if "SOURCE_BASELINE_METHOD_KEY" in globals() and SOURCE_BASELINE_METHOD_KEY
    else CANDIDATE_RETRIEVAL_METHOD_KEY
)
stale_source_method_label = (
    SOURCE_BASELINE_METHOD_LABEL
    if "SOURCE_BASELINE_METHOD_LABEL" in globals() and SOURCE_BASELINE_METHOD_LABEL
    else CANDIDATE_RETRIEVAL_METHOD_LABEL
)
stale_terms = [
    f"tuned_{stale_source_method_key}",
    f"baseline_tuned_{stale_source_method_key}",
    f"Tuned {stale_source_method_label} Retrieval",
]

search_payload = json.dumps(make_jsonable({
    'config_snapshot': config_snapshot if 'config_snapshot' in globals() else {},
    'run_manifest': run_manifest if 'run_manifest' in globals() else {},
}), ensure_ascii=False)

stale_found = [term for term in stale_terms if term in search_payload]
if stale_found:
    raise RuntimeError(f"Stale tuned graph-hybrid metadata found in Notebook 10 outputs: {stale_found}")

print('Saved outputs:')
for path in saved_paths:
    print('-', path)

# Refresh the exported contracts with Notebook 09 winner lineage.
notebook09_lineage = {
    "notebook_09_dependency": True,
    "stage1_candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_role": CANDIDATE_POOL_ROLE,
    "candidate_retrieval_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "candidate_retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "source_baseline_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "source_baseline_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "candidate_budget_policy": stage1_pool_manifest.get("candidate_budget_policy"),
    "candidate_budget_k": int(stage1_pool_manifest.get("candidate_budget_k", POOL_K)),
    "exact_k_validation_passed": not bool(stage1_pool_manifest.get("variable_candidate_count_allowed", True)),
    "user_prior_feature_policy": USER_PRIOR_FEATURE_POLICY,
    "user_prior_features_enabled": bool(USE_USER_PRIOR_FEATURES),
    "strict_prior_history_loaded": bool(USE_USER_PRIOR_FEATURES),
    "prior_history_loaded": bool(USE_USER_PRIOR_FEATURES),
    "candidate_source_qc": candidate_source_qc,
}
config_snapshot.update(notebook09_lineage)
Path(config_snapshot_path).write_text(
    json.dumps(make_jsonable(config_snapshot), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
run_manifest.update(notebook09_lineage)
run_manifest.setdefault("input_paths", {})["stage1_candidate_pool_manifest"] = str(CANDIDATE_POOL_MANIFEST_PATH)
run_manifest["input_paths"]["winner_candidate_pool"] = str(CANDIDATE_POOL_PATH)
run_manifest["input_paths"]["query_cache_parquet"] = str(QUERY_CACHE_PARQUET)
if not USE_USER_PRIOR_FEATURES:
    run_manifest["input_paths"]["prior_history_parquet"] = None
Path(run_manifest_path).write_text(
    json.dumps(make_jsonable(run_manifest), ensure_ascii=False, indent=2),
    encoding="utf-8",
)


In [ ]:
# =========================================================
# Feature Manifest, Temporal Summary, and Leakage QC Export
# =========================================================
feature_manifest_path = OUT_DIR / 'feature_manifest.json'
used_model_features_path = OUT_DIR / 'used_model_features.csv'
feature_leakage_qc_path = OUT_DIR / 'feature_leakage_qc.csv'
temporal_feature_summary_path = OUT_DIR / 'temporal_feature_summary.csv'

used_model_features_df.to_csv(used_model_features_path, index=False)
feature_leakage_qc_df.to_csv(feature_leakage_qc_path, index=False)
temporal_feature_summary_df.to_csv(temporal_feature_summary_path, index=False)
Path(feature_manifest_path).write_text(json.dumps(make_jsonable(feature_manifest), ensure_ascii=False, indent=2), encoding='utf-8')

for path_obj in [feature_manifest_path, used_model_features_path, feature_leakage_qc_path, temporal_feature_summary_path]:
    if 'saved_paths' in globals() and path_obj not in saved_paths:
        saved_paths.append(path_obj)

if 'config_snapshot_path' in globals() and Path(config_snapshot_path).exists():
    config_snapshot_existing = load_json_if_exists(config_snapshot_path)
    config_snapshot_existing.update({
        'feature_registry_version': SHANNON_FEATURE_REGISTRY_VERSION,
        'temporal_feature_version': TEMPORAL_FEATURE_VERSION,
        'temporal_leakage_rule': TEMPORAL_LEAKAGE_RULE,
        'shannon_temporal_recency_weight': float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
        'feature_manifest_path': str(feature_manifest_path),
        'used_model_features_path': str(used_model_features_path),
        'feature_leakage_qc_path': str(feature_leakage_qc_path),
        'temporal_feature_summary_path': str(temporal_feature_summary_path),
    })
    Path(config_snapshot_path).write_text(json.dumps(make_jsonable(config_snapshot_existing), ensure_ascii=False, indent=2), encoding='utf-8')

if 'run_manifest_path' in globals() and Path(run_manifest_path).exists():
    run_manifest_existing = load_json_if_exists(run_manifest_path)
    run_manifest_existing['feature_parity'] = {
        'baseline_type': 'deterministic_heuristic',
        'feature_registry_version': SHANNON_FEATURE_REGISTRY_VERSION,
        'temporal_feature_version': TEMPORAL_FEATURE_VERSION,
        'temporal_leakage_rule': TEMPORAL_LEAKAGE_RULE,
        'temporal_recency_features_included': False,
        'ablation_condition': 'S2-Q',
        'no_prior_feature_policy': globals().get('NO_PRIOR_FEATURE_POLICY', ''),
        'no_prior_manifest_note': NO_PRIOR_MANIFEST_NOTE,
        'no_prior_model_feature_contract_clean': bool(no_prior_model_feature_contract_clean),
        'no_prior_score_components_disabled': bool(no_prior_score_components_disabled),
        'no_prior_shannon_ranking_matches_stage1_baseline': bool(no_prior_shannon_ranking_matches_stage1_baseline),
        'raw_timestamp_columns_excluded': True,
        'note': NO_PRIOR_MANIFEST_NOTE,
    }
    run_manifest_existing.setdefault('output_paths', {})
    run_manifest_existing['output_paths'].update({
        'feature_manifest': str(feature_manifest_path),
        'used_model_features': str(used_model_features_path),
        'feature_leakage_qc': str(feature_leakage_qc_path),
        'temporal_feature_summary': str(temporal_feature_summary_path),
    })
    run_manifest_existing['created_outputs'] = list(dict.fromkeys(run_manifest_existing.get('created_outputs', []) + [
        str(feature_manifest_path),
        str(used_model_features_path),
        str(feature_leakage_qc_path),
        str(temporal_feature_summary_path),
    ]))
    Path(run_manifest_path).write_text(json.dumps(make_jsonable(run_manifest_existing), ensure_ascii=False, indent=2), encoding='utf-8')

print('Feature contract outputs saved:')
for path_obj in [feature_manifest_path, used_model_features_path, feature_leakage_qc_path, temporal_feature_summary_path]:
    print('-', path_obj)

# Persist the executed brand/parity contract in the existing manifests.
if Path(config_snapshot_path).exists():
    config_contract = load_json_if_exists(config_snapshot_path)
    config_contract.update({
        **brand_contract,
        "scoring_components": list(SHANNON_SCORING_COMPONENTS),
        "feature_columns": list(SHANNON_FEATURE_COLUMNS),
        "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
        "contract_diagnostics": contract_diagnostics,
    })
    Path(config_snapshot_path).write_text(json.dumps(make_jsonable(config_contract), ensure_ascii=False, indent=2), encoding="utf-8")

if Path(run_manifest_path).exists():
    run_contract = load_json_if_exists(run_manifest_path)
    run_contract.update({
        **brand_contract,
        "scoring_components": list(SHANNON_SCORING_COMPONENTS),
        "feature_columns": list(SHANNON_FEATURE_COLUMNS),
        "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
        "contract_diagnostics": contract_diagnostics,
        "shared_all_prior_contract_path": str(SHARED_ALL_PRIOR_CONTRACT_PATH) if USE_USER_PRIOR_FEATURES else None,
    })
    Path(run_manifest_path).write_text(json.dumps(make_jsonable(run_contract), ensure_ascii=False, indent=2), encoding="utf-8")


## 10. Final validation

In [ ]:
expected_outputs = [
    results_overall_path,
    results_by_pool_depth_path,
    results_by_regime_path,
    results_by_regime_pool_depth_path,
    runtime_by_pool_depth_path,
    uplift_summary_path,
    feature_table_path,
    per_query_results_path,
    reranked_candidates_path,
    query_profiles_path,
    OUT_DIR / 'runtime_steps.csv',
    OUT_DIR / 'runtime_notebook_summary.csv',
    OUT_DIR / 'runtime_method_summary_at1000.csv',
    OUT_DIR / 'runtime_pool_depth_diagnostic.csv',
    OUT_DIR / 'runtime_method_components_at1000_face.csv',
    config_snapshot_path,
    diagnostics_summary_path,
    run_manifest_path,
]
missing_outputs = [str(path) for path in expected_outputs if not Path(path).exists()]
if missing_outputs:
    raise RuntimeError(f'Missing expected outputs: {missing_outputs}')

if sorted(summary_overall_df['rerank_method'].astype(str).unique().tolist()) != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError('results_overall methods do not match EXPECTED_RERANK_METHODS.')
if sorted(per_query_results_df['rerank_method'].astype(str).unique().tolist()) != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError('per_query_metrics methods do not match EXPECTED_RERANK_METHODS.')
if sorted(pd.to_numeric(per_query_results_df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist()) != sorted(POOL_DEPTHS):
    raise RuntimeError('per_query_metrics does not preserve all configured POOL_DEPTHS.')
if candidate_set_changed_by_reranker:
    raise RuntimeError('Candidate set was changed by Shannon reranker.')

print('Final validation passed.')
print('Primary Stage 2 metric:', PRIMARY_STAGE2_METRIC)
display(summary_overall_df[['rerank_method', 'n_queries', 'NDCG@5', 'HitRate@5', 'MRR@5'] + [c for c in ['weighted_facet_overlap_at_5'] if c in summary_overall_df.columns]])
display(runtime_method_summary_at1000_df)

In [ ]:
# =========================================================
# Notebook 09 Winner-Lineage QC
# =========================================================
expected_path_key = (
    "query_only_winner_long"
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else "personalized_winner_long"
)
expected_candidate_path = Path(stage1_pool_manifest["output_paths"][expected_path_key])
expected_candidate_method_key = (
    stage1_pool_manifest["baseline_retrieval_winner_method_key"]
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else stage1_pool_manifest["selected_personalized_method_slug"]
)
expected_candidate_method_label = (
    stage1_pool_manifest["baseline_retrieval_winner_method_label"]
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else stage1_pool_manifest["selected_personalized_method_label"]
)

winner_lineage_qc_df = pd.DataFrame([
    {
        "check": "candidate_path_matches_notebook09",
        "passed": Path(CANDIDATE_POOL_PATH) == expected_candidate_path,
        "observed": str(CANDIDATE_POOL_PATH),
        "expected": str(expected_candidate_path),
    },
    {
        "check": "candidate_role_matches_condition",
        "passed": candidate_df["candidate_pool_role"].astype(str).eq(CANDIDATE_POOL_ROLE).all(),
        "observed": ",".join(sorted(candidate_df["candidate_pool_role"].astype(str).unique().tolist())),
        "expected": CANDIDATE_POOL_ROLE,
    },
    {
        "check": "candidate_method_key_matches_notebook09",
        "passed": CANDIDATE_RETRIEVAL_METHOD_KEY == expected_candidate_method_key,
        "observed": CANDIDATE_RETRIEVAL_METHOD_KEY,
        "expected": expected_candidate_method_key,
    },
    {
        "check": "candidate_method_label_matches_notebook09",
        "passed": CANDIDATE_RETRIEVAL_METHOD_LABEL == expected_candidate_method_label,
        "observed": CANDIDATE_RETRIEVAL_METHOD_LABEL,
        "expected": expected_candidate_method_label,
    },
    {
        "check": "exact_k_candidate_count",
        "passed": candidate_df.groupby("query_id")["item_id"].size().eq(POOL_K).all(),
        "observed": f"{candidate_df.groupby('query_id')['item_id'].size().min()}..{candidate_df.groupby('query_id')['item_id'].size().max()}",
        "expected": str(POOL_K),
    },
    {
        "check": "user_prior_policy_matches_condition",
        "passed": bool(USE_USER_PRIOR_FEATURES) == (EXPERIMENT_CONDITION != "s2q_no_prior_reranking"),
        "observed": str(bool(USE_USER_PRIOR_FEATURES)),
        "expected": str(EXPERIMENT_CONDITION != "s2q_no_prior_reranking"),
    },
])

winner_lineage_qc_path = OUT_DIR / f"winner_candidate_source_qc_{CATEGORY_ID}.csv"
winner_lineage_qc_df.to_csv(winner_lineage_qc_path, index=False)
print("Winner candidate source QC:")
display(winner_lineage_qc_df)
print("Output:", winner_lineage_qc_path)

if not winner_lineage_qc_df["passed"].all():
    failed = winner_lineage_qc_df.loc[~winner_lineage_qc_df["passed"], "check"].tolist()
    raise RuntimeError(f"Notebook 09 winner-lineage QC failed: {failed}")


In [ ]:
# =========================================================
# Runtime depth QC
# =========================================================

expected_runtime_depths = sorted(int(depth) for depth in POOL_DEPTHS)
observed_runtime_depths = (
    sorted(pd.to_numeric(runtime_by_pool_depth_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and "pool_depth" in runtime_by_pool_depth_df.columns
    else []
)
runtime_depths_complete = observed_runtime_depths == expected_runtime_depths

runtime_required_cols = [
    "pool_depth",
    "candidate_pool_depth",
    "n_queries",
    "n_candidates",
    "feature_preparation_runtime_sec",
    "model_scoring_runtime_sec",
    "ranking_sorting_runtime_sec",
    "online_operation_runtime_sec",
    "runtime_sec_per_query",
    "runtime_sec_per_candidate",
    "queries_per_second",
    "candidates_per_second",
]
runtime_missing_cols = [
    col for col in runtime_required_cols
    if "runtime_by_pool_depth_df" not in globals()
    or not isinstance(runtime_by_pool_depth_df, pd.DataFrame)
    or col not in runtime_by_pool_depth_df.columns
]

runtime_duplicate_rows = 0
if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
    duplicate_key_cols = [col for col in ["pool_depth", "method", "reranker", "reranking_method"] if col in runtime_by_pool_depth_df.columns]
    if duplicate_key_cols:
        runtime_duplicate_rows = int(runtime_by_pool_depth_df.duplicated(duplicate_key_cols).sum())

runtime_file_candidates = []

for _name in [
    "runtime_by_pool_depth_path",
    "runtime_report_summary_path",
    "runtime_steps_path",
    "runtime_notebook_summary_path",
]:
    if _name in globals():
        runtime_file_candidates.append(globals()[_name])

if "output_paths" in globals() and isinstance(output_paths, dict):
    runtime_file_candidates.extend(
        path for key, path in output_paths.items()
        if str(key).startswith("runtime") or str(key).endswith("manifest")
    )

runtime_files_created = sorted({
    str(Path(path)) for path in runtime_file_candidates
    if path is not None and Path(path).exists()
})


print("Candidate pool depths evaluated:", expected_runtime_depths)
print("Candidate pool depths in runtime_by_pool_depth.csv:", observed_runtime_depths)
print("Every evaluated pool depth has a runtime row:", runtime_depths_complete)
print("Output runtime files created:")
for path in runtime_files_created:
    print(path)
runtime_all_null_components = []
if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
    for col in [
        "feature_preparation_runtime_sec",
        "model_scoring_runtime_sec",
        "ranking_sorting_runtime_sec",
        "online_operation_runtime_sec",
    ]:
        if col in runtime_by_pool_depth_df.columns and pd.to_numeric(runtime_by_pool_depth_df[col], errors="coerce").notna().sum() == 0:
            runtime_all_null_components.append(col)
if runtime_missing_cols:
    print("Runtime QC warning: missing runtime columns:", runtime_missing_cols)
else:
    print("Runtime QC missing columns: none")
if runtime_all_null_components:
    print("Runtime QC warning: runtime components not measured per depth:", runtime_all_null_components)
else:
    print("Runtime QC unmeasured components: none")
if runtime_duplicate_rows:
    print("Runtime QC warning: duplicate runtime rows:", runtime_duplicate_rows)
else:
    print("Runtime QC duplicate rows: none")


In [ ]:
# =========================================================
# Metric depth QC
# =========================================================

expected_pool_depths = [int(depth) for depth in POOL_DEPTHS]
expected_eval_ks = [int(k) for k in EVAL_KS]
required_metric_cols = [
    f"{metric}@{k}"
    for metric in ["HitRate", "NDCG", "MRR"]
    for k in expected_eval_ks
]

metric_source_df = None
for candidate_name in [
    "per_query_results_df",
    "per_query_final_eval_df",
    "per_query_metrics_df",
    "summary_by_pool_depth_df",
    "results_by_pool_depth_df",
]:
    candidate_df = globals().get(candidate_name)
    if isinstance(candidate_df, pd.DataFrame) and not candidate_df.empty:
        metric_source_df = candidate_df
        metric_source_name = candidate_name
        break
if metric_source_df is None:
    metric_source_df = pd.DataFrame()
    metric_source_name = "none"

available_hitrate_cols = [col for col in required_metric_cols if col.startswith("HitRate@") and col in metric_source_df.columns]
available_ndcg_cols = [col for col in required_metric_cols if col.startswith("NDCG@") and col in metric_source_df.columns]
available_mrr_cols = [col for col in required_metric_cols if col.startswith("MRR@") and col in metric_source_df.columns]
missing_metric_cols = [col for col in required_metric_cols if col not in metric_source_df.columns]

pool_metric_df = globals().get("summary_by_pool_depth_df")
if not isinstance(pool_metric_df, pd.DataFrame) or pool_metric_df.empty:
    pool_metric_df = globals().get("results_by_pool_depth_df")
if not isinstance(pool_metric_df, pd.DataFrame) or pool_metric_df.empty:
    pool_metric_df = metric_source_df

observed_pool_depths = (
    sorted(pd.to_numeric(pool_metric_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    if isinstance(pool_metric_df, pd.DataFrame) and "pool_depth" in pool_metric_df.columns
    else []
)
pool_depth_metric_missing = {}
if isinstance(pool_metric_df, pd.DataFrame) and "pool_depth" in pool_metric_df.columns:
    for depth in expected_pool_depths:
        depth_rows = pool_metric_df[pd.to_numeric(pool_metric_df["pool_depth"], errors="coerce").eq(depth)]
        missing_for_depth = [col for col in required_metric_cols if col not in depth_rows.columns]
        if depth_rows.empty or missing_for_depth:
            pool_depth_metric_missing[int(depth)] = missing_for_depth if missing_for_depth else ["no rows"]
else:
    pool_depth_metric_missing = {int(depth): ["pool_depth column missing"] for depth in expected_pool_depths}
all_pool_depths_have_metrics = not pool_depth_metric_missing and observed_pool_depths == expected_pool_depths

output_file_candidates = []
if "saved_paths" in globals():
    output_file_candidates.extend(saved_paths)
if "expected_outputs" in globals():
    output_file_candidates.extend(expected_outputs)
if "output_paths" in globals() and isinstance(output_paths, dict):
    output_file_candidates.extend(output_paths.values())
for name, value in list(globals().items()):
    if name.endswith("_path") and any(token in name for token in ["results", "runtime", "manifest", "summary", "metrics", "candidates"]):
        output_file_candidates.append(value)

output_files_created = []
for path_value in output_file_candidates:
    try:
        path_obj = Path(path_value)
    except TypeError:
        continue
    if path_obj.exists() and str(path_obj) not in output_files_created:
        output_files_created.append(str(path_obj))

print("POOL_DEPTHS:", expected_pool_depths)
print("EVAL_KS:", expected_eval_ks)
print("Metric source dataframe:", metric_source_name)
print("Available HitRate columns:", available_hitrate_cols)
print("Available NDCG columns:", available_ndcg_cols)
print("Available MRR columns:", available_mrr_cols)
print("Every evaluated pool depth has all required EVAL_KS metrics:", all_pool_depths_have_metrics)
if missing_metric_cols:
    print("Missing metric columns:", missing_metric_cols)
if pool_depth_metric_missing:
    print("Pool-depth metric gaps:", pool_depth_metric_missing)
print("Output files created:")
for path in output_files_created:
    print(path)
